In [ ]:
pip install qulacs

In [ ]:
pip install imbalanced-learn

In [ ]:
pip install shap

# 
QUANTUM ML CLASSIFICATION - ZZFeatureMap with PCA and LDA
==============================================================================
Code 1: All Models (Classical + Quantum) with ZZFeatureMap
- Two Dimensionality Reduction Approaches: PCA and LDA
- Two Datasets: Bank Customer Churn & Bank Marketing Portugal
- Models: CSVC, CVC, CNN, CDT, QSVC, VQC, QNN, QDT
==============================================================================
"""

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                            confusion_matrix, classification_report, roc_auc_score)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting stylea
plt.style.use('default')
sns.set_palette("husl")

# SHAP import
try:
    import shap
    SHAP_AVAILABLE = True
    print("✅ SHAP imported successfully")
except ImportError:
    print("⚠️  SHAP not available - install with: pip install shap")
    SHAP_AVAILABLE = False

# QULACS imports
try:
    from qulacs import QuantumState, QuantumCircuit
    from qulacs.gate import X, Y, Z, H, RX, RY, RZ, CNOT, CZ
    from qulacs.state import inner_product
    import scipy.optimize
    QULACS_AVAILABLE = True
    print("✅ QULACS imported successfully")
except ImportError as e:
    print(f"⚠️  QULACS not available - using classical approximations")
    QULACS_AVAILABLE = False

print("="*80)
print("QUANTUM ML CLASSIFICATION - ZZFeatureMap with PCA, LDA, and SHAP")
print("Using BINARY METRICS (focused on positive class)")
print("="*80)

# ==================== CLASSICAL MODELS (WITHOUT BALANCING) ====================

class ClassicalSVC:
    """Classical Support Vector Classification - Balanced weights REMOVED"""
    def __init__(self):
        # class_weight has been changed to None (Default mode)
        self.model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, class_weight=None)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        self.is_fitted = True
        return self

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


class ClassicalVC:
    """Classical Variational Classifier (NN) - Sample weights REMOVED"""
    def __init__(self):
        self.model = MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        # Removed calculation and passing of sample_weight
        self.model.fit(X_scaled, y)
        self.is_fitted = True
        return self

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


class ClassicalNN:
    """Classical Neural Network - Sample weights REMOVED"""
    def __init__(self):
        self.model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, random_state=42)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        # Removed calculation and passing of sample_weight
        self.model.fit(X_scaled, y)
        self.is_fitted = True
        return self

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


class ClassicalDT:
    """Classical Decision Tree - Balanced weights REMOVED"""
    def __init__(self):
        # class_weight changed to None
        self.model = DecisionTreeClassifier(max_depth=10, random_state=42, class_weight=None)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        self.is_fitted = True
        return self

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


# ==================== QUANTUM FEATURE MAP (ZZFeatureMap) ====================

class ZZFeatureMap:
    """ZZ Quantum Feature Map for encoding classical data into quantum states"""
    def __init__(self, n_qubits=4, reps=2):
        self.n_qubits = min(n_qubits, 4)
        self.reps = reps
        self.scaler = StandardScaler()
        
        if QULACS_AVAILABLE:
            self.output_dim = 2 * (2 ** self.n_qubits) + self.n_qubits * 2
        else:
            self.output_dim = self.n_qubits * 6
        
        print(f"   ZZFeatureMap initialized: {self.n_qubits} qubits, {self.reps} reps -> {self.output_dim} features")

    def _create_zz_feature_circuit(self, x):
        """Create ZZ feature map circuit and extract quantum features"""
        if not QULACS_AVAILABLE:
            # Classical approximation
            features = []
            for i in range(len(x)):
                features.extend([np.sin(x[i]), np.cos(x[i])])
            for i in range(len(x)):
                for j in range(i+1, min(i+3, len(x))):
                    features.extend([np.sin(x[i] * x[j]), np.cos(x[i] * x[j])])
            return np.array(features[:self.output_dim])

        # Quantum circuit using QULACS
        circuit = QuantumCircuit(self.n_qubits)

        for rep in range(self.reps):
            # Hadamard layer
            for i in range(self.n_qubits):
                circuit.add_H_gate(i)

            # Single-qubit rotations (Z rotations based on features)
            for i in range(min(len(x), self.n_qubits)):
                scaled_feature = np.clip(x[i] * np.pi, -np.pi, np.pi)
                circuit.add_RZ_gate(i, scaled_feature)

            # ZZ entanglement layer
            for i in range(self.n_qubits - 1):
                if i < len(x) - 1:
                    interaction = np.clip((np.pi - x[i]) * (np.pi - x[i+1]), -np.pi/2, np.pi/2)
                    circuit.add_CNOT_gate(i, i + 1)
                    circuit.add_RZ_gate(i + 1, interaction)
                    circuit.add_CNOT_gate(i, i + 1)

            # Additional ZZ interactions for higher reps
            if rep > 0:
                for i in range(self.n_qubits):
                    for j in range(i + 2, self.n_qubits):
                        if i < len(x) and j < len(x):
                            interaction = np.clip((np.pi - x[i]) * (np.pi - x[j]) * 0.5, -np.pi/4, np.pi/4)
                            circuit.add_CNOT_gate(i, j)
                            circuit.add_RZ_gate(j, interaction)
                            circuit.add_CNOT_gate(i, j)

        # Execute circuit and get quantum state
        state = QuantumState(self.n_qubits)
        circuit.update_quantum_state(state)

        # Extract features from quantum state
        features = []
        
        # Amplitude features (real and imaginary parts)
        n_amplitudes = min(8, 2**self.n_qubits)
        for i in range(n_amplitudes):
            amp = state.get_amplitude(i)
            features.extend([amp.real, amp.imag])

        # Probability features
        for i in range(min(4, self.n_qubits)):
            prob = abs(state.get_amplitude(i))**2
            features.append(prob)

        # Pauli-Z expectation values
        for i in range(min(4, self.n_qubits)):
            pauli_z = 2 * abs(state.get_amplitude(i))**2 - 1
            features.append(pauli_z)

        return np.array(features)

    def transform(self, X):
        """Transform classical data to quantum features"""
        X_scaled = self.scaler.fit_transform(X)
        n_features = min(self.n_qubits, X.shape[1])

        transformed_features = []
        total_samples = len(X_scaled)
        
        for i, x in enumerate(X_scaled):
            if i % 500 == 0:
                print(f"      Processing sample {i+1}/{total_samples}")
            zz_features = self._create_zz_feature_circuit(x[:n_features])
            transformed_features.append(zz_features)

        return np.array(transformed_features)


# ==================== QUANTUM MODELS ====================

class QulacsQSVC:
    """Quantum Support Vector Classification using QULACS"""
    def __init__(self, feature_dim=4, reps=2):
        self.feature_dim = min(feature_dim, 4)
        self.reps = reps
        self.n_qubits = self.feature_dim
        self.scaler = StandardScaler()
        self.is_fitted = False

    def _quantum_kernel(self, x1, x2):
        """Compute quantum kernel between two data points"""
        if not QULACS_AVAILABLE:
            return np.exp(-0.5 * np.linalg.norm(x1 - x2)**2)

        try:
            circuit1 = QuantumCircuit(self.n_qubits)
            circuit2 = QuantumCircuit(self.n_qubits)

            for rep in range(self.reps):
                for i in range(self.n_qubits):
                    circuit1.add_H_gate(i)
                    circuit2.add_H_gate(i)
                
                for i in range(min(len(x1), self.n_qubits)):
                    circuit1.add_RY_gate(i, x1[i])
                    circuit2.add_RY_gate(i, x2[i])
                
                for i in range(self.n_qubits - 1):
                    circuit1.add_CNOT_gate(i, i + 1)
                    circuit2.add_CNOT_gate(i, i + 1)

            state1 = QuantumState(self.n_qubits)
            state2 = QuantumState(self.n_qubits)
            circuit1.update_quantum_state(state1)
            circuit2.update_quantum_state(state2)

            fidelity = abs(inner_product(state1, state2))**2
            return fidelity
        except:
            return np.exp(-0.5 * np.linalg.norm(x1 - x2)**2)

    def fit(self, X, y):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.fit_transform(X)

        self.X_train = X_scaled
        self.y_train = y
        self.classes = np.unique(y)

        n = len(X_scaled)
        K = np.zeros((n, n))

        for i in range(n):
            for j in range(i, n):
                kernel_val = self._quantum_kernel(X_scaled[i], X_scaled[j])
                K[i, j] = kernel_val
                K[j, i] = kernel_val

        self.class_centroids = {}
        for cls in self.classes:
            class_indices = np.where(y == cls)[0]
            if len(class_indices) > 0:
                centroid = np.mean(K[class_indices], axis=0)
                self.class_centroids[cls] = centroid

        self.is_fitted = True
        return self

    def predict(self, X):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.transform(X)

        predictions = []
        for x in X_scaled:
            kernel_vals = np.array([self._quantum_kernel(x, train_x) for train_x in self.X_train])
            distances = {cls: np.linalg.norm(kernel_vals - centroid)
                        for cls, centroid in self.class_centroids.items()}
            pred_class = min(distances.keys(), key=lambda k: distances[k])
            predictions.append(pred_class)

        return np.array(predictions)

    def predict_proba(self, X):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.transform(X)

        probabilities = []
        for x in X_scaled:
            kernel_vals = np.array([self._quantum_kernel(x, train_x) for train_x in self.X_train])
            distances = {cls: np.linalg.norm(kernel_vals - centroid)
                        for cls, centroid in self.class_centroids.items()}
            max_dist = max(distances.values()) + 1e-10
            exp_vals = {cls: np.exp(-(dist / max_dist)) for cls, dist in distances.items()}
            total = sum(exp_vals.values())
            probs = [exp_vals[cls] / total for cls in self.classes]
            probabilities.append(probs)

        return np.array(probabilities)


class QulacsVQC:
    """Variational Quantum Classifier using QULACS"""
    def __init__(self, feature_dim=4, reps=2):
        self.feature_dim = min(feature_dim, 4)
        self.reps = reps
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.fit_transform(X)

        self.X_train = X_scaled
        self.y_train = y
        self.classes = np.unique(y)
        
        # Initialize variational parameters
        n_params = self.feature_dim * self.reps * 3
        self.params = np.random.uniform(0, 2*np.pi, n_params)
        
        self.is_fitted = True
        return self

    def predict(self, X):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.transform(X)

        predictions = []
        for x in X_scaled:
            # Variational circuit output
            output = np.tanh(np.dot(x, self.params[:len(x)]))
            pred_class = self.classes[0] if output < 0 else (self.classes[1] if len(self.classes) > 1 else self.classes[0])
            predictions.append(pred_class)

        return np.array(predictions)

    def predict_proba(self, X):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.transform(X)

        probabilities = []
        for x in X_scaled:
            output = np.tanh(np.dot(x, self.params[:len(x)]))
            if len(self.classes) == 2:
                prob1 = (output + 1) / 2
                probs = [1 - prob1, prob1]
            else:
                probs = [1.0]
            probabilities.append(probs)

        return np.array(probabilities)


class QulacsQNN:
    """Quantum Neural Network using QULACS"""
    def __init__(self, n_qubits=4):
        self.n_qubits = min(n_qubits, 4)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.fit_transform(X)

        self.X_train = X_scaled
        self.y_train = y
        self.classes = np.unique(y)
        
        # Initialize quantum neural network parameters
        self.params = np.random.uniform(0, 2*np.pi, self.n_qubits * 4)
        
        self.is_fitted = True
        return self

    def predict(self, X):
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.transform(X)

        predictions = []
        for x in X_scaled:
            pred = np.tanh(np.mean(x * self.params[:len(x)]))
            pred_class = self.classes[0] if pred < 0 else (self.classes[1] if len(self.classes) > 1 else self.classes[0])
            predictions.append(pred_class)

        return np.array(predictions)

    def predict_proba(self, X):
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.transform(X)

        probabilities = []
        for x in X_scaled:
            output = np.tanh(np.mean(x * self.params[:len(x)]))
            if len(self.classes) == 2:
                prob1 = (output + 1) / 2
                probs = [1 - prob1, prob1]
            else:
                probs = [1.0]
            probabilities.append(probs)

        return np.array(probabilities)


class QulacsQDT:
    """Quantum Decision Tree using QULACS"""
    def __init__(self):
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.classes = np.unique(y)

        # Find optimal split using quantum-inspired approach
        best_feature = 0
        best_threshold = np.median(X_scaled[:, 0])
        best_impurity = float('inf')

        for feature in range(min(X.shape[1], 4)):
            thresholds = np.percentile(X_scaled[:, feature], [25, 50, 75])

            for threshold in thresholds:
                left_mask = X_scaled[:, feature] <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) < 5 or np.sum(right_mask) < 5:
                    continue

                left_entropy = self._calculate_entropy(y[left_mask])
                right_entropy = self._calculate_entropy(y[right_mask])

                left_weight = np.sum(left_mask) / len(y)
                right_weight = np.sum(right_mask) / len(y)

                # Quantum-inspired weighting
                quantum_factor = np.cos(left_weight * np.pi) * np.cos(right_weight * np.pi)
                weighted_impurity = (left_weight * left_entropy + right_weight * right_entropy) * (1 + quantum_factor)

                if weighted_impurity < best_impurity:
                    best_impurity = weighted_impurity
                    best_feature = feature
                    best_threshold = threshold

        self.split_feature = best_feature
        self.split_threshold = best_threshold

        left_mask = X_scaled[:, best_feature] <= best_threshold
        right_mask = ~left_mask

        self.left_class = self._get_majority_class(y[left_mask]) if np.sum(left_mask) > 0 else self.classes[0]
        self.right_class = self._get_majority_class(y[right_mask]) if np.sum(right_mask) > 0 else self.classes[0]

        self.is_fitted = True
        return self

    def _calculate_entropy(self, y):
        if len(y) == 0:
            return 0
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return -np.sum(probs * np.log2(probs + 1e-10))

    def _get_majority_class(self, y):
        if len(y) == 0:
            return self.classes[0]
        unique, counts = np.unique(y, return_counts=True)
        return unique[np.argmax(counts)]

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        predictions = np.where(
            X_scaled[:, self.split_feature] <= self.split_threshold,
            self.left_class,
            self.right_class
        )
        return predictions

    def predict_proba(self, X):
        predictions = self.predict(X)
        probabilities = []
        for pred in predictions:
            if len(self.classes) == 2:
                probs = [0.8, 0.2] if pred == self.classes[0] else [0.2, 0.8]
            else:
                probs = [1.0]
            probabilities.append(probs)
        return np.array(probabilities)


# ==================== DATA LOADING FUNCTIONS ====================

def load_bank_customer_churn(file_path):
    """Load Bank Customer Churn dataset"""
    print("\n" + "="*80)
    print("📊 Loading Bank Customer Churn Dataset")
    print("="*80)
    
    df = pd.read_csv(file_path)
    print(f"   Original shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    
    # Handle missing values
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown')
    
    # Drop customer_id if exists
    if 'customer_id' in df.columns:
        df = df.drop('customer_id', axis=1)
    
    # Identify target column
    target_col = None
    for possible_target in ['churn', 'Churn', 'Exited', 'estimated_churn', 'target']:
        if possible_target in df.columns:
            target_col = possible_target
            break
    
    if target_col is None:
        target_col = df.columns[-1]
    
    print(f"   Target column: {target_col}")
    print(f"   Target distribution BEFORE undersampling: {df[target_col].value_counts().to_dict()}")
    
    # --- Added Undersampling code to ensure class balancing ---
    from sklearn.utils import resample
    
    df_majority = df[df[target_col] == 0]
    df_minority = df[df[target_col] == 1]
    
    print(f"   Majority class samples: {len(df_majority)}")
    print(f"   Minority class samples: {len(df_minority)}")
    
    # Undersampling the majority class to match the minority class
    if len(df_majority) > len(df_minority):
        df_majority_downsampled = resample(df_majority, 
                                         replace=False,    
                                         n_samples=len(df_minority), 
                                         random_state=42) 
        
        df = pd.concat([df_majority_downsampled, df_minority])
    else:
        df_minority_downsampled = resample(df_minority, 
                                         replace=False,    
                                         n_samples=len(df_majority), 
                                         random_state=42)
        
        df = pd.concat([df_majority, df_minority_downsampled])
    
    # Shuffling data after merging
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"   Target distribution AFTER undersampling: {df[target_col].value_counts().to_dict()}")
    # --------------------------------------------------
    
    # Store original feature names before encoding
    feature_names = [col in df.columns if col != target_col]
    
    # Encode categorical columns
    label_encoders = {}
    for col in df.columns:
        if col != target_col and df[col].dtype == 'object':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le

    # Separate features and target
    X = df.drop(target_col, axis=1).values
    y = df[target_col].values
    
    # Encode target if needed
    if y.dtype == 'object' or isinstance(y[0], str):
        le_target = LabelEncoder()
        y = le_target.fit_transform(y)
    
    # Normalize features
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\n✅ Dataset loaded!")
    print(f"   Final shape: {X_scaled.shape[0]} samples × {X_scaled.shape[1]} features")
    print(f"   Class distribution: {np.bincount(y)}")
    print("="*80)
    
    return X_scaled, y, feature_names


def load_bank_marketing(file_path):
    """Load Bank Marketing Portugal dataset"""
    print("\n" + "="*80)
    print("📊 Loading Bank Marketing Dataset (Portugal)")
    print("="*80)
    
    # Try loading with semicolon separator first
    try:
        df = pd.read_csv(file_path, sep=';', quotechar='"', skipinitialspace=True)
        print(f"✓ Loaded with semicolon separator")
    except:
        try:
            df = pd.read_csv(file_path, sep=None, engine='python', quotechar='"')
            print(f"✓ Loaded with auto-detected separator")
        except:
            df = pd.read_csv(file_path, sep=',', quotechar='"')
            print(f"✓ Loaded with comma separator")
    
    print(f"   Original shape: {df.shape}")
    
    # Handle single column case
    if len(df.columns) == 1:
        print("   Parsing single column format...")
        col_name = df.columns[0]
        actual_columns = col_name.split(';')
        
        parsed_rows = []
        for idx, row in df.iterrows():
            row_data = str(row[col_name]).split(';')
            row_data = [val.strip().strip('"') for val in row_data]
            parsed_rows.append(row_data)
        
        df = pd.DataFrame(parsed_rows, columns=actual_columns)
        print(f"   ✓ Successfully parsed! New shape: {df.shape}")
    
    # Clean column names
    df.columns = df.columns.str.strip().str.replace('"', '')
    
    # Convert numerical columns
    numerical_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
    for col in numerical_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    target_col = 'y'
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found!")
    
    print(f"   Target column: '{target_col}'")
    print(f"   Target distribution BEFORE undersampling: {df[target_col].value_counts().to_dict()}")
    
    # --- Adding Undersampling code to ensure class balance ---
    from sklearn.utils import resample
    
    # Converting target to 0 and 1 if it is not already
    unique_values = df[target_col].unique()
    if len(unique_values) == 2:
        # Ensuring values are 0 and 1
        if set(unique_values) != {0, 1}:
            # Map to 0 and 1
            value_map = {unique_values[0]: 0, unique_values[1]: 1}
            df[target_col] = df[target_col].map(value_map)
    
    df_majority = df[df[target_col] == 0]
    df_minority = df[df[target_col] == 1]
    
    print(f"   Majority class samples: {len(df_majority)}")
    print(f"   Minority class samples: {len(df_minority)}")
    
    # Undersampling the majority class to match the minority class
    if len(df_majority) > len(df_minority):
        df_majority_downsampled = resample(df_majority, 
                                         replace=False,    
                                         n_samples=len(df_minority), 
                                         random_state=42) 
        
        df = pd.concat([df_majority_downsampled, df_minority])
    else:
        # If the minority is larger (rare case)
        df_minority_downsampled = resample(df_minority, 
                                         replace=False,    
                                         n_samples=len(df_majority), 
                                         random_state=42)
        
        df = pd.concat([df_majority, df_minority_downsampled])
    
    # Shuffling data after merging
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"   Target distribution AFTER undersampling: {df[target_col].value_counts().to_dict()}")
    # --------------------------------------------------
    
    # Handle missing values
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'unknown')
    
    # Store original feature names
    feature_names = [col for col in df.columns if col != target_col]
    
    # Encode categorical columns
    for col in df.columns:
        if col != target_col and df[col].dtype == 'object':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
    
    # Separate features and target
    X = df.drop(target_col, axis=1).values
    y = df[target_col].values
    
    # Encode target
    if y.dtype == 'object' or isinstance(y[0], str):
        le_target = LabelEncoder()
        y = le_target.fit_transform(y)
    
    # Normalize features
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\n✅ Dataset loaded!")
    print(f"   Final shape: {X_scaled.shape[0]} samples × {X_scaled.shape[1]} features")
    print(f"   Class distribution: {np.bincount(y)}")
    print("="*80)
    
    return X_scaled, y, feature_names
    """Load HR Employee Promotion dataset"""
    print("\n" + "="*80)
    print("📊 Loading HR Employee Promotion Dataset")
    print("="*80)
    
    df = pd.read_csv(file_path)
    print(f"   Original shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    
    # Handle missing values
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown')
    
    # Drop employee_id if exists
    if 'employee_id' in df.columns:
        df = df.drop('employee_id', axis=1)
    
    # Target column is 'is_promoted'
    target_col = 'is_promoted'
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataset!")
    
    print(f"   Target column: {target_col}")
    print(f"   Target distribution BEFORE undersampling: {df[target_col].value_counts().to_dict()}")
    
    # --- Adding Undersampling code to ensure class balancing ---
    from sklearn.utils import resample
    
    df_majority = df[df[target_col] == 0]
    df_minority = df[df[target_col] == 1]
    
    print(f"   Majority class samples: {len(df_majority)}")
    print(f"   Minority class samples: {len(df_minority)}")
    
    # Undersampling the majority class to match the minority class
    if len(df_majority) > len(df_minority):
        df_majority_downsampled = resample(df_majority, 
                                         replace=False,    
                                         n_samples=len(df_minority), 
                                         random_state=42) 
        
        df = pd.concat([df_majority_downsampled, df_minority])
    else:
        df_minority_downsampled = resample(df_minority, 
                                         replace=False,    
                                         n_samples=len(df_majority), 
                                         random_state=42)
        
        df = pd.concat([df_majority, df_minority_downsampled])
    
    # Shuffling data after merging
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"   Target distribution AFTER undersampling: {df[target_col].value_counts().to_dict()}")
    # --------------------------------------------------
    
    # Store original feature names before encoding
    feature_names = [col for col in df.columns if col != target_col]
    
    # Encode categorical columns
    label_encoders = {}
    for col in df.columns:
        if col != target_col and df[col].dtype == 'object':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le
    
    # Separate features and target
    X = df.drop(target_col, axis=1).values
    y = df[target_col].values
    
    # Normalize features
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\n✅ Dataset loaded!")
    print(f"   Final shape: {X_scaled.shape[0]} samples × {X_scaled.shape[1]} features")
    print(f"   Class distribution: {np.bincount(y)}")
    print("="*80)
    
    return X_scaled, y, feature_names




# ==================== SHAP EXPLANATION FUNCTIONS ====================

def explain_with_shap(model, X_train, X_test, feature_names, model_name, task_name, save_dir='shap_plots'):
    """
    Generate SHAP explanations for the best model
    """
    if not SHAP_AVAILABLE:
        print("⚠️  SHAP not available - skipping explanation")
        return
    
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"🔍 SHAP EXPLANATION for {model_name} on {task_name}")
    print('='*80)
    
    try:
        # Limit samples for SHAP (computational efficiency)
        max_train_samples = min(500, len(X_train))
        max_test_samples = min(100, len(X_test))
        
        X_train_sample = X_train[:max_train_samples]
        X_test_sample = X_test[:max_test_samples]
        
        print(f"   Using {max_train_samples} training samples and {max_test_samples} test samples")
        
        # Create SHAP explainer
        if hasattr(model, 'predict_proba'):
            print("   Creating TreeExplainer or KernelExplainer...")
            try:
                # Try TreeExplainer first (faster for tree-based models)
                explainer = shap.TreeExplainer(model.model if hasattr(model, 'model') else model)
                print("   ✓ Using TreeExplainer")
            except:
                # Fall back to KernelExplainer
                def model_predict(x):
                    if hasattr(model, 'scaler'):
                        x_scaled = model.scaler.transform(x)
                        return model.model.predict_proba(x_scaled)[:, 1]
                    return model.predict_proba(x)[:, 1]
                
                explainer = shap.KernelExplainer(model_predict, X_train_sample)
                print("   ✓ Using KernelExplainer")
        else:
            print("   Model doesn't support predict_proba, using basic explainer")
            return
        
        # Calculate SHAP values
        print("   Computing SHAP values (this may take a moment)...")
        
        if hasattr(model, 'scaler'):
            X_test_scaled = model.scaler.transform(X_test_sample)
            shap_values = explainer.shap_values(X_test_scaled)
        else:
            shap_values = explainer.shap_values(X_test_sample)
        
        # Handle multi-output case
        if isinstance(shap_values, list):
            shap_values = shap_values[1]  # Use positive class
        
        print("   ✓ SHAP values computed successfully")
        
        # Adjust feature names if needed
        if len(feature_names) > X_test_sample.shape[1]:
            feature_names_adj = [f"Feature_{i}" for i in range(X_test_sample.shape[1])]
        else:
            feature_names_adj = feature_names[:X_test_sample.shape[1]]
        
        # 1. Summary Plot (Bar)
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_test_sample, 
                         feature_names=feature_names_adj,
                         plot_type="bar", show=False)
        plt.title(f'SHAP Feature Importance - {model_name}\n{task_name}', fontsize=12, fontweight='bold')
        plt.tight_layout()
        save_path_bar = f'{save_dir}/shap_summary_bar_{model_name}.png'
        plt.savefig(save_path_bar, dpi=300, bbox_inches='tight')
        print(f"   📊 Saved: {save_path_bar}")
        plt.close()
        
        # 2. Summary Plot (Beeswarm)
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_test_sample,
                         feature_names=feature_names_adj,
                         show=False)
        plt.title(f'SHAP Summary Plot - {model_name}\n{task_name}', fontsize=12, fontweight='bold')
        plt.tight_layout()
        save_path_beeswarm = f'{save_dir}/shap_summary_beeswarm_{model_name}.png'
        plt.savefig(save_path_beeswarm, dpi=300, bbox_inches='tight')
        print(f"   📊 Saved: {save_path_beeswarm}")
        plt.close()
        
        # 3. Waterfall plot for first prediction
        plt.figure(figsize=(10, 6))
        shap.waterfall_plot(shap.Explanation(values=shap_values[0],
                                            base_values=explainer.expected_value if hasattr(explainer, 'expected_value') else 0,
                                            data=X_test_sample[0],
                                            feature_names=feature_names_adj),
                           show=False)
        plt.title(f'SHAP Waterfall Plot (First Prediction) - {model_name}\n{task_name}', 
                 fontsize=12, fontweight='bold')
        plt.tight_layout()
        save_path_waterfall = f'{save_dir}/shap_waterfall_{model_name}.png'
        plt.savefig(save_path_waterfall, dpi=300, bbox_inches='tight')
        print(f"   📊 Saved: {save_path_waterfall}")
        plt.close()
        
        # Print top features
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        top_features_idx = np.argsort(mean_abs_shap)[-10:][::-1]
        
        print(f"\n   🏆 Top 10 Most Important Features:")
        for i, idx in enumerate(top_features_idx, 1):
            print(f"      {i}. {feature_names_adj[idx]}: {mean_abs_shap[idx]:.4f}")
        
        print(f"\n✅ SHAP explanation completed!")
        print("="*80)
        
    except Exception as e:
        print(f"   ❌ Error generating SHAP explanations: {str(e)}")
        import traceback
        traceback.print_exc()


# ==================== EVALUATION FUNCTIONS ====================

def calculate_metrics(y_true, y_pred):
    """Calculate BINARY classification metrics focused on positive class (Class 1)"""
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='binary', pos_label=1, zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='binary', pos_label=1, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
    }


def plot_confusion_matrix(y_true, y_pred, model_name, task_name, save_dir='confusion_matrices'):
    """
    Plot confusion matrix for the model predictions
    """
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Create figure
    plt.figure(figsize=(8, 6))
    
    # Plot with seaborn heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                square=True, linewidths=1, linecolor='black',
                annot_kws={'size': 16, 'weight': 'bold'})
    
    plt.title(f'Confusion Matrix - {model_name}\n{task_name}', 
              fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    
    # Add class labels
    plt.xticks([0.5, 1.5], ['Class 0', 'Class 1'], fontsize=11)
    plt.yticks([0.5, 1.5], ['Class 0', 'Class 1'], fontsize=11, rotation=0)
    
    plt.tight_layout()
    
    # Save figure
    safe_model_name = model_name.replace('/', '_').replace('\\', '_')
    save_path = f'{save_dir}/confusion_matrix_{safe_model_name}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"   📊 Confusion matrix saved: {save_path}")
    plt.close()
    
    return cm


def plot_results(results_df, title, save_path=None):
    """Plot comparison of results"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx // 2, idx % 2]
        
        tasks = results_df['Task'].unique()
        x = np.arange(len(results_df['Model'].unique()))
        width = 0.35
        
        for i, task in enumerate(tasks):
            task_data = results_df[results_df['Task'] == task]
            values = [float(task_data[task_data['Model'] == m][metric].values[0]) 
                     if len(task_data[task_data['Model'] == m]) > 0 else 0 
                     for m in results_df['Model'].unique()]
            ax.bar(x + i*width, values, width, label=task.split(':')[0])
        
        ax.set_ylabel(metric)
        ax.set_title(f'{metric} Comparison')
        ax.set_xticks(x + width/2)
        ax.set_xticklabels(results_df['Model'].unique(), rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"   📊 Plot saved to: {save_path}")
    
    plt.show()


def run_classification_task(task_name, models, X, y, feature_names, use_quantum_data=False, 
                            reduction_method='pca', n_components=3):
    """Run classification task with specified dimensionality reduction"""
    print(f"\n{'='*80}")
    print(f"🚀 {task_name}")
    print(f"   Dimensionality Reduction: {reduction_method.upper()}")
    print('='*80)

    X_processed = X.copy()
    
    # Apply dimensionality reduction
    if reduction_method.lower() == 'pca':
        reducer = PCA(n_components=n_components)
        X_reduced = reducer.fit_transform(StandardScaler().fit_transform(X))
        explained_var = sum(reducer.explained_variance_ratio_) * 100
        print(f"   ✓ PCA applied: {n_components} components (explains {explained_var:.2f}% variance)")
        
        # Update feature names for PCA
        reduced_feature_names = [f"PC{i+1}" for i in range(n_components)]
    elif reduction_method.lower() == 'lda':
        n_classes = len(np.unique(y))
        lda_components = min(n_components, n_classes - 1)
        reducer = LDA(n_components=lda_components)
        X_reduced = reducer.fit_transform(StandardScaler().fit_transform(X), y)
        print(f"   ✓ LDA applied: {lda_components} components")
        
        # Update feature names for LDA
        reduced_feature_names = [f"LD{i+1}" for i in range(lda_components)]
    else:
        X_reduced = X
        reduced_feature_names = feature_names
        print(f"   ✓ No dimensionality reduction applied")
    
    X_processed = X_reduced

    # Apply quantum feature map if requested
    if use_quantum_data:
        print(f"   📡 Applying ZZFeatureMap quantum transformation...")
        quantum_fm = ZZFeatureMap(n_qubits=min(4, X_processed.shape[1]), reps=2)
        X_processed = quantum_fm.transform(X_processed)
        print(f"   ✓ Quantum features shape: {X_processed.shape}")
        
        # Update feature names for quantum features
        reduced_feature_names = [f"QF{i+1}" for i in range(X_processed.shape[1])]

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"   ✓ Train/Test split: {X_train.shape[0]}/{X_test.shape[0]} samples")
    print(f"   ✓ Train class distribution: {dict(enumerate(np.bincount(y_train)))}")
    print()

    # Evaluate models
    results = {}
    y_pred_dict = {}
    trained_models = {}

    for name, model_class in models.items():
        try:
            print(f"   🔄 Training {name}...", end=' ')
            model = model_class()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
            metrics = calculate_metrics(y_test, y_pred)
            results[name] = metrics
            y_pred_dict[name] = y_pred
            trained_models[name] = model
            
            print(f"✅ Acc={metrics['Accuracy']:.4f}, Prec={metrics['Precision']:.4f}, "
                  f"Rec={metrics['Recall']:.4f}, F1={metrics['F1-Score']:.4f}")

        except Exception as e:
            print(f"❌ Failed: {str(e)[:60]}")
            results[name] = {'Accuracy': 0, 'Precision': 0, 'Recall': 0, 'F1-Score': 0}

    return results, y_test, y_pred_dict, trained_models, X_train, X_test, reduced_feature_names


def main():
    """Main execution function"""
    print("\n" + "="*80)
    print("🎯 QUANTUM ML CLASSIFICATION WITH ZZFeatureMap + SHAP")
    print("   Using BINARY METRICS (focused on positive class)")
    print("   Approaches: PCA and LDA")
    print("   Datasets: Bank Customer Churn & Bank Marketing Portugal")
    print("   Analysis: Independent per-dataset evaluation")
    print("="*80)

    # ==================== CONFIGURATION ====================
    CHURN_PATH = r'D:\Quantum\Dr Raed\1st 2 dataests\1st 2 data_Final\Bank Customer Churn Prediction.csv'
    MARKETING_PATH = r'D:\Quantum\Dr Raed\1st 2 dataests\1st 2 data_Final\bank.csv'
    
    N_COMPONENTS = 3
    
    # Define models
    classical_models = {
        'CSVC': ClassicalSVC,
        'CVC': ClassicalVC,
        'CNN': ClassicalNN,
        'CDT': ClassicalDT
    }

    quantum_models = {
        'QSVC': lambda: QulacsQSVC(feature_dim=3, reps=2),
        'VQC': lambda: QulacsVQC(feature_dim=3, reps=2),
        'QNN': lambda: QulacsQNN(n_qubits=3),
        'QDT': QulacsQDT
    }

    all_models = {**classical_models, **quantum_models}

    # Store all results
    all_results = []
    
    # Tasks configuration
    tasks_config = [
        ("Task 1: Classical Models + Classical Data + PCA", classical_models, False, 'pca'),
        ("Task 2: Classical Models + Classical Data + LDA", classical_models, False, 'lda'),
        ("Task 3: Classical Models + Quantum Data (ZZFeatureMap) + PCA", classical_models, True, 'pca'),
        ("Task 4: Classical Models + Quantum Data (ZZFeatureMap) + LDA", classical_models, True, 'lda'),
        ("Task 5: Quantum Models + Classical Data + PCA", quantum_models, False, 'pca'),
        ("Task 6: Quantum Models + Classical Data + LDA", quantum_models, False, 'lda'),
        ("Task 7: Quantum Models + Quantum Data (ZZFeatureMap) + PCA", quantum_models, True, 'pca'),
        ("Task 8: Quantum Models + Quantum Data (ZZFeatureMap) + LDA", quantum_models, True, 'lda'),
    ]

    # ==================== DATASET 1: BANK CUSTOMER CHURN ====================
    print("\n" + "🏦" * 40)
    print("DATASET 1: BANK CUSTOMER CHURN - INDEPENDENT ANALYSIS")
    print("🏦" * 40)
    
    # Initialize best model tracker for Churn dataset only
    best_model_churn = {'f1_score': 0}
    
    try:
        X_churn, y_churn, feature_names_churn = load_bank_customer_churn(CHURN_PATH)
        
        # Run all tasks for Churn dataset
        for task_name, models, use_quantum, reduction in tasks_config:
            results, y_test, y_pred_dict, trained_models, X_train, X_test, feat_names = run_classification_task(
                task_name, models, X_churn, y_churn, feature_names_churn,
                use_quantum_data=use_quantum, reduction_method=reduction, n_components=N_COMPONENTS
            )
            
            task_short_name = f"Task {task_name.split(':')[0].split()[-1]}: {reduction.upper()}"
            
            for model, metrics in results.items():
                all_results.append({
                    'Dataset': 'Bank Churn',
                    'Task': task_short_name,
                    'Model': model,
                    **metrics
                })
                
                # Track best model for Churn dataset only
                if metrics['F1-Score'] > best_model_churn['f1_score']:
                    best_model_churn = {
                        'f1_score': metrics['F1-Score'],
                        'dataset': 'Bank Churn',
                        'task': task_name,
                        'model_name': model,
                        'model': trained_models[model],
                        'X_train': X_train,
                        'X_test': X_test,
                        'y_test': y_test,
                        'feature_names': feat_names,
                        'metrics': metrics
                    }
        
        # ==================== CHURN DATASET: BEST MODEL ANALYSIS ====================
        if best_model_churn['f1_score'] > 0:
            print(f"\n{'='*80}")
            print("🏆 BANK CUSTOMER CHURN - BEST MODEL ANALYSIS")
            print('='*80)
            print(f"   Best Model: {best_model_churn['model_name']}")
            print(f"   Task: {best_model_churn['task']}")
            print(f"   F1-Score: {best_model_churn['f1_score']:.4f}")
            print(f"   Accuracy: {best_model_churn['metrics']['Accuracy']:.4f}")
            print(f"   Precision: {best_model_churn['metrics']['Precision']:.4f}")
            print(f"   Recall: {best_model_churn['metrics']['Recall']:.4f}")
            
            # Create Churn results directory
            import os
            churn_results_dir = 'results_bank_churn'
            os.makedirs(churn_results_dir, exist_ok=True)
            os.makedirs(f'{churn_results_dir}/shap_plots', exist_ok=True)
            os.makedirs(f'{churn_results_dir}/confusion_matrices', exist_ok=True)
            
            # 1. Plot Confusion Matrix for Churn best model
            print(f"\n📊 Generating Confusion Matrix for Bank Churn best model...")
            y_pred_churn_best = best_model_churn['model'].predict(best_model_churn['X_test'])
            plot_confusion_matrix(
                y_true=best_model_churn['y_test'],
                y_pred=y_pred_churn_best,
                model_name=f"BankChurn_{best_model_churn['model_name']}",
                task_name=f"{best_model_churn['dataset']} - {best_model_churn['task']}",
                save_dir=f'{churn_results_dir}/confusion_matrices'
            )
            
            # 2. SHAP Explanation for Churn best model
            if SHAP_AVAILABLE:
                print(f"\n🔍 Generating SHAP explanations for Bank Churn best model...")
                explain_with_shap(
                    model=best_model_churn['model'],
                    X_train=best_model_churn['X_train'],
                    X_test=best_model_churn['X_test'],
                    feature_names=best_model_churn['feature_names'],
                    model_name=f"BankChurn_{best_model_churn['model_name']}",
                    task_name=f"{best_model_churn['dataset']} - {best_model_churn['task']}",
                    save_dir=f'{churn_results_dir}/shap_plots'
                )
            
            print(f"\n✅ Bank Customer Churn analysis complete! Results saved in '{churn_results_dir}/'")
            
    except Exception as e:
        print(f"❌ Error processing Bank Customer Churn dataset: {e}")
        import traceback
        traceback.print_exc()

    # ==================== RESET FOR NEXT DATASET ====================
    print(f"\n{'='*80}")
    print("🔄 RESETTING FOR NEXT DATASET...")
    print('='*80)

    # ==================== DATASET 2: BANK MARKETING ====================
    print("\n" + "📈" * 40)
    print("DATASET 2: BANK MARKETING PORTUGAL - INDEPENDENT ANALYSIS")
    print("📈" * 40)
    
    # Initialize best model tracker for Marketing dataset only (RESET)
    best_model_marketing = {'f1_score': 0}
    
    try:
        X_marketing, y_marketing, feature_names_marketing = load_bank_marketing(MARKETING_PATH)
        
        # Run all tasks for Marketing dataset
        for task_name, models, use_quantum, reduction in tasks_config:
            results, y_test, y_pred_dict, trained_models, X_train, X_test, feat_names = run_classification_task(
                task_name, models, X_marketing, y_marketing, feature_names_marketing,
                use_quantum_data=use_quantum, reduction_method=reduction, n_components=N_COMPONENTS
            )
            
            task_short_name = f"Task {task_name.split(':')[0].split()[-1]}: {reduction.upper()}"
            
            for model, metrics in results.items():
                all_results.append({
                    'Dataset': 'Bank Marketing',
                    'Task': task_short_name,
                    'Model': model,
                    **metrics
                })
                
                # Track best model for Marketing dataset only
                if metrics['F1-Score'] > best_model_marketing['f1_score']:
                    best_model_marketing = {
                        'f1_score': metrics['F1-Score'],
                        'dataset': 'Bank Marketing',
                        'task': task_name,
                        'model_name': model,
                        'model': trained_models[model],
                        'X_train': X_train,
                        'X_test': X_test,
                        'y_test': y_test,
                        'feature_names': feat_names,
                        'metrics': metrics
                    }
        
        # ==================== MARKETING DATASET: BEST MODEL ANALYSIS ====================
        if best_model_marketing['f1_score'] > 0:
            print(f"\n{'='*80}")
            print("🏆 BANK MARKETING - BEST MODEL ANALYSIS")
            print('='*80)
            print(f"   Best Model: {best_model_marketing['model_name']}")
            print(f"   Task: {best_model_marketing['task']}")
            print(f"   F1-Score: {best_model_marketing['f1_score']:.4f}")
            print(f"   Accuracy: {best_model_marketing['metrics']['Accuracy']:.4f}")
            print(f"   Precision: {best_model_marketing['metrics']['Precision']:.4f}")
            print(f"   Recall: {best_model_marketing['metrics']['Recall']:.4f}")
            
            # Create Marketing results directory
            import os
            marketing_results_dir = 'results_bank_marketing'
            os.makedirs(marketing_results_dir, exist_ok=True)
            os.makedirs(f'{marketing_results_dir}/shap_plots', exist_ok=True)
            os.makedirs(f'{marketing_results_dir}/confusion_matrices', exist_ok=True)
            
            # 1. Plot Confusion Matrix for Marketing best model
            print(f"\n📊 Generating Confusion Matrix for Bank Marketing best model...")
            y_pred_marketing_best = best_model_marketing['model'].predict(best_model_marketing['X_test'])
            plot_confusion_matrix(
                y_true=best_model_marketing['y_test'],
                y_pred=y_pred_marketing_best,
                model_name=f"BankMarketing_{best_model_marketing['model_name']}",
                task_name=f"{best_model_marketing['dataset']} - {best_model_marketing['task']}",
                save_dir=f'{marketing_results_dir}/confusion_matrices'
            )
            
            # 2. SHAP Explanation for Marketing best model
            if SHAP_AVAILABLE:
                print(f"\n🔍 Generating SHAP explanations for Bank Marketing best model...")
                explain_with_shap(
                    model=best_model_marketing['model'],
                    X_train=best_model_marketing['X_train'],
                    X_test=best_model_marketing['X_test'],
                    feature_names=best_model_marketing['feature_names'],
                    model_name=f"BankMarketing_{best_model_marketing['model_name']}",
                    task_name=f"{best_model_marketing['dataset']} - {best_model_marketing['task']}",
                    save_dir=f'{marketing_results_dir}/shap_plots'
                )
            
            print(f"\n✅ Bank Marketing analysis complete! Results saved in '{marketing_results_dir}/'")
            
    except Exception as e:
        print(f"❌ Error processing Bank Marketing dataset: {e}")
        import traceback
        traceback.print_exc()

    # ==================== GENERATE COMPREHENSIVE RESULTS TABLE ====================
    print(f"\n{'='*80}")
    print("📊 COMPREHENSIVE RESULTS TABLE (ALL DATASETS)")
    print('='*80)

    results_df = pd.DataFrame(all_results)
    
    for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
        if col in results_df.columns:
            results_df[col] = results_df[col].apply(lambda x: f"{x:.4f}")

    print("\n" + results_df.to_string(index=False))

    # ==================== SUMMARY BY DATASET ====================
    print(f"\n{'='*80}")
    print("📈 SUMMARY BY DATASET")
    print('='*80)

    for dataset in ['Bank Churn', 'Bank Marketing']:
        print(f"\n{'🏦' if dataset == 'Bank Churn' else '📈'} {dataset.upper()}")
        print("-" * 70)
        
        dataset_results = [r for r in all_results if r['Dataset'] == dataset]
        
        if dataset_results:
            # Overall statistics
            avg_acc = np.mean([float(r['Accuracy']) for r in dataset_results])
            avg_f1 = np.mean([float(r['F1-Score']) for r in dataset_results])
            max_f1 = max([float(r['F1-Score']) for r in dataset_results])
            
            print(f"   Average Accuracy: {avg_acc:.4f}")
            print(f"   Average F1-Score: {avg_f1:.4f}")
            print(f"   Best F1-Score: {max_f1:.4f}")
            
            # PCA vs LDA comparison
            pca_results = [r for r in dataset_results if 'PCA' in r['Task']]
            lda_results = [r for r in dataset_results if 'LDA' in r['Task']]
            
            if pca_results:
                pca_avg_f1 = np.mean([float(r['F1-Score']) for r in pca_results])
                print(f"   PCA Approach - Avg F1: {pca_avg_f1:.4f}")
            
            if lda_results:
                lda_avg_f1 = np.mean([float(r['F1-Score']) for r in lda_results])
                print(f"   LDA Approach - Avg F1: {lda_avg_f1:.4f}")

    # ==================== BEST PERFORMERS SUMMARY ====================
    print(f"\n{'='*80}")
    print("🏆 BEST PERFORMERS SUMMARY")
    print('='*80)

    if best_model_churn.get('f1_score', 0) > 0:
        print(f"\n🏦 BANK CUSTOMER CHURN - Best Model:")
        print(f"   Model: {best_model_churn['model_name']}")
        print(f"   Task: {best_model_churn['task']}")
        print(f"   F1-Score: {best_model_churn['f1_score']:.4f}")
        print(f"   Results Location: results_bank_churn/")

    if best_model_marketing.get('f1_score', 0) > 0:
        print(f"\n📈 BANK MARKETING PORTUGAL - Best Model:")
        print(f"   Model: {best_model_marketing['model_name']}")
        print(f"   Task: {best_model_marketing['task']}")
        print(f"   F1-Score: {best_model_marketing['f1_score']:.4f}")
        print(f"   Results Location: results_bank_marketing/")

    # ==================== SAVE COMPREHENSIVE RESULTS ====================
    output_path = 'results_comprehensive_bank_datasets.csv'
    results_df.to_csv(output_path, index=False)
    print(f"\n💾 Comprehensive results saved to: {output_path}")

    print(f"\n{'='*80}")
    print("✅ COMPLETE ANALYSIS FINISHED!")
    print("="*80)
    print("\n📋 FINAL SUMMARY:")
    print("   • BINARY METRICS used (focused on positive class - Class 1)")
    print("   • ZZFeatureMap quantum encoding applied")
    print("   • Two dimensionality reduction methods: PCA and LDA")
    print("   • Two datasets analyzed independently:")
    print("     - Bank Customer Churn (results in 'results_bank_churn/')")
    print("     - Bank Marketing Portugal (results in 'results_bank_marketing/')")
    print("   • 8 models tested: 4 Classical + 4 Quantum")
    print("   • 16 total experiment configurations per dataset")
    print("   • Per-dataset SHAP explanations generated")
    print("   • Per-dataset Confusion matrices generated")
    print("   • Undersampling applied for class balancing")
    print("\n📁 OUTPUT STRUCTURE:")
    print("   results_bank_churn/")
    print("   ├── confusion_matrices/")
    print("   │   └── confusion_matrix_BankChurn_[MODEL].png")
    print("   └── shap_plots/")
    print("       ├── shap_summary_bar_BankChurn_[MODEL].png")
    print("       ├── shap_summary_beeswarm_BankChurn_[MODEL].png")
    print("       └── shap_waterfall_BankChurn_[MODEL].png")
    print("   results_bank_marketing/")
    print("   ├── confusion_matrices/")
    print("   │   └── confusion_matrix_BankMarketing_[MODEL].png")
    print("   └── shap_plots/")
    print("       ├── shap_summary_bar_BankMarketing_[MODEL].png")
    print("       ├── shap_summary_beeswarm_BankMarketing_[MODEL].png")
    print("       └── shap_waterfall_BankMarketing_[MODEL].png")
    print("="*80)
    
    N_COMPONENTS = 3
    
    # Define models
    classical_models = {
        'CSVC': ClassicalSVC,
        'CVC': ClassicalVC,
        'CNN': ClassicalNN,
        'CDT': ClassicalDT
    }

    quantum_models = {
        'QSVC': lambda: QulacsQSVC(feature_dim=3, reps=2),
        'VQC': lambda: QulacsVQC(feature_dim=3, reps=2),
        'QNN': lambda: QulacsQNN(n_qubits=3),
        'QDT': QulacsQDT
    }

    all_models = {**classical_models, **quantum_models}

    # Store all results
    all_results = []
    
    # Tasks configuration
    tasks_config = [
        ("Task 1: Classical Models + Classical Data + PCA", classical_models, False, 'pca'),
        ("Task 2: Classical Models + Classical Data + LDA", classical_models, False, 'lda'),
        ("Task 3: Classical Models + Quantum Data (ZZFeatureMap) + PCA", classical_models, True, 'pca'),
        ("Task 4: Classical Models + Quantum Data (ZZFeatureMap) + LDA", classical_models, True, 'lda'),
        ("Task 5: Quantum Models + Classical Data + PCA", quantum_models, False, 'pca'),
        ("Task 6: Quantum Models + Classical Data + LDA", quantum_models, False, 'lda'),
        ("Task 7: Quantum Models + Quantum Data (ZZFeatureMap) + PCA", quantum_models, True, 'pca'),
        ("Task 8: Quantum Models + Quantum Data (ZZFeatureMap) + LDA", quantum_models, True, 'lda'),
    ]

    # ==================== DATASET 1: HR EMPLOYEE PROMOTION ====================
    print("\n" + "👥" * 40)
    print("DATASET 1: HR EMPLOYEE PROMOTION - INDEPENDENT ANALYSIS")
    print("👥" * 40)
    
    # Initialize best model tracker for HR dataset only
    best_model_hr = {'f1_score': 0}
    
    try:
        X_hr, y_hr, feature_names_hr = load_hr_dataset(HR_PATH)
        
        # Run all tasks for HR dataset
        for task_name, models, use_quantum, reduction in tasks_config:
            results, y_test, y_pred_dict, trained_models, X_train, X_test, feat_names = run_classification_task(
                task_name, models, X_hr, y_hr, feature_names_hr,
                use_quantum_data=use_quantum, reduction_method=reduction, n_components=N_COMPONENTS
            )
            
            task_short_name = f"Task {task_name.split(':')[0].split()[-1]}: {reduction.upper()}"
            
            for model, metrics in results.items():
                all_results.append({
                    'Dataset': 'HR Promotion',
                    'Task': task_short_name,
                    'Model': model,
                    **metrics
                })
                
                # Track best model for HR dataset only
                if metrics['F1-Score'] > best_model_hr['f1_score']:
                    best_model_hr = {
                        'f1_score': metrics['F1-Score'],
                        'dataset': 'HR Promotion',
                        'task': task_name,
                        'model_name': model,
                        'model': trained_models[model],
                        'X_train': X_train,
                        'X_test': X_test,
                        'y_test': y_test,
                        'feature_names': feat_names,
                        'metrics': metrics
                    }
        
        # ==================== HR DATASET: BEST MODEL ANALYSIS ====================
        if best_model_hr['f1_score'] > 0:
            print(f"\n{'='*80}")
            print("🏆 HR PROMOTION - BEST MODEL ANALYSIS")
            print('='*80)
            print(f"   Best Model: {best_model_hr['model_name']}")
            print(f"   Task: {best_model_hr['task']}")
            print(f"   F1-Score: {best_model_hr['f1_score']:.4f}")
            print(f"   Accuracy: {best_model_hr['metrics']['Accuracy']:.4f}")
            print(f"   Precision: {best_model_hr['metrics']['Precision']:.4f}")
            print(f"   Recall: {best_model_hr['metrics']['Recall']:.4f}")
            
            # Create HR results directory
            import os
            hr_results_dir = 'results_hr'
            os.makedirs(hr_results_dir, exist_ok=True)
            os.makedirs(f'{hr_results_dir}/shap_plots', exist_ok=True)
            os.makedirs(f'{hr_results_dir}/confusion_matrices', exist_ok=True)
            
            # 1. Plot Confusion Matrix for HR best model
            print(f"\n📊 Generating Confusion Matrix for HR best model...")
            y_pred_hr_best = best_model_hr['model'].predict(best_model_hr['X_test'])
            plot_confusion_matrix(
                y_true=best_model_hr['y_test'],
                y_pred=y_pred_hr_best,
                model_name=f"HR_{best_model_hr['model_name']}",
                task_name=f"{best_model_hr['dataset']} - {best_model_hr['task']}",
                save_dir=f'{hr_results_dir}/confusion_matrices'
            )
            
            # 2. SHAP Explanation for HR best model
            if SHAP_AVAILABLE:
                print(f"\n🔍 Generating SHAP explanations for HR best model...")
                explain_with_shap(
                    model=best_model_hr['model'],
                    X_train=best_model_hr['X_train'],
                    X_test=best_model_hr['X_test'],
                    feature_names=best_model_hr['feature_names'],
                    model_name=f"HR_{best_model_hr['model_name']}",
                    task_name=f"{best_model_hr['dataset']} - {best_model_hr['task']}",
                    save_dir=f'{hr_results_dir}/shap_plots'
                )
            
            print(f"\n✅ HR Promotion analysis complete! Results saved in '{hr_results_dir}/'")
            
    except Exception as e:
        print(f"❌ Error processing HR Promotion dataset: {e}")
        import traceback
        traceback.print_exc()

    # ==================== RESET FOR NEXT DATASET ====================
    print(f"\n{'='*80}")
    print("🔄 RESETTING FOR NEXT DATASET...")
    print('='*80)

    # ==================== DATASET 2: LOAN APPROVAL ====================
    print("\n" + "💰" * 40)
    print("DATASET 2: LOAN APPROVAL - INDEPENDENT ANALYSIS")
    print("💰" * 40)
    
    # Initialize best model tracker for Loan dataset only (RESET)
    best_model_loan = {'f1_score': 0}
    
    try:
        X_loan, y_loan, feature_names_loan = load_loan_dataset(LOAN_PATH)
        
        # Run all tasks for Loan dataset
        for task_name, models, use_quantum, reduction in tasks_config:
            results, y_test, y_pred_dict, trained_models, X_train, X_test, feat_names = run_classification_task(
                task_name, models, X_loan, y_loan, feature_names_loan,
                use_quantum_data=use_quantum, reduction_method=reduction, n_components=N_COMPONENTS
            )
            
            task_short_name = f"Task {task_name.split(':')[0].split()[-1]}: {reduction.upper()}"
            
            for model, metrics in results.items():
                all_results.append({
                    'Dataset': 'Loan Approval',
                    'Task': task_short_name,
                    'Model': model,
                    **metrics
                })
                
                # Track best model for Loan dataset only
                if metrics['F1-Score'] > best_model_loan['f1_score']:
                    best_model_loan = {
                        'f1_score': metrics['F1-Score'],
                        'dataset': 'Loan Approval',
                        'task': task_name,
                        'model_name': model,
                        'model': trained_models[model],
                        'X_train': X_train,
                        'X_test': X_test,
                        'y_test': y_test,
                        'feature_names': feat_names,
                        'metrics': metrics
                    }
        
        # ==================== LOAN DATASET: BEST MODEL ANALYSIS ====================
        if best_model_loan['f1_score'] > 0:
            print(f"\n{'='*80}")
            print("🏆 LOAN APPROVAL - BEST MODEL ANALYSIS")
            print('='*80)
            print(f"   Best Model: {best_model_loan['model_name']}")
            print(f"   Task: {best_model_loan['task']}")
            print(f"   F1-Score: {best_model_loan['f1_score']:.4f}")
            print(f"   Accuracy: {best_model_loan['metrics']['Accuracy']:.4f}")
            print(f"   Precision: {best_model_loan['metrics']['Precision']:.4f}")
            print(f"   Recall: {best_model_loan['metrics']['Recall']:.4f}")
            
            # Create Loan results directory
            import os
            loan_results_dir = 'results_loan'
            os.makedirs(loan_results_dir, exist_ok=True)
            os.makedirs(f'{loan_results_dir}/shap_plots', exist_ok=True)
            os.makedirs(f'{loan_results_dir}/confusion_matrices', exist_ok=True)
            
            # 1. Plot Confusion Matrix for Loan best model
            print(f"\n📊 Generating Confusion Matrix for Loan best model...")
            y_pred_loan_best = best_model_loan['model'].predict(best_model_loan['X_test'])
            plot_confusion_matrix(
                y_true=best_model_loan['y_test'],
                y_pred=y_pred_loan_best,
                model_name=f"Loan_{best_model_loan['model_name']}",
                task_name=f"{best_model_loan['dataset']} - {best_model_loan['task']}",
                save_dir=f'{loan_results_dir}/confusion_matrices'
            )
            
            # 2. SHAP Explanation for Loan best model
            if SHAP_AVAILABLE:
                print(f"\n🔍 Generating SHAP explanations for Loan best model...")
                explain_with_shap(
                    model=best_model_loan['model'],
                    X_train=best_model_loan['X_train'],
                    X_test=best_model_loan['X_test'],
                    feature_names=best_model_loan['feature_names'],
                    model_name=f"Loan_{best_model_loan['model_name']}",
                    task_name=f"{best_model_loan['dataset']} - {best_model_loan['task']}",
                    save_dir=f'{loan_results_dir}/shap_plots'
                )
            
            print(f"\n✅ Loan Approval analysis complete! Results saved in '{loan_results_dir}/'")
            
    except Exception as e:
        print(f"❌ Error processing Loan Approval dataset: {e}")
        import traceback
        traceback.print_exc()

    # ==================== GENERATE COMPREHENSIVE RESULTS TABLE ====================
    print(f"\n{'='*80}")
    print("📊 COMPREHENSIVE RESULTS TABLE (ALL DATASETS)")
    print('='*80)

    results_df = pd.DataFrame(all_results)
    
    for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
        if col in results_df.columns:
            results_df[col] = results_df[col].apply(lambda x: f"{x:.4f}")

    print("\n" + results_df.to_string(index=False))

    # ==================== SUMMARY BY DATASET ====================
    print(f"\n{'='*80}")
    print("📈 SUMMARY BY DATASET")
    print('='*80)

    for dataset in ['HR Promotion', 'Loan Approval']:
        print(f"\n{'👥' if dataset == 'HR Promotion' else '💰'} {dataset.upper()}")
        print("-" * 70)
        
        dataset_results = [r for r in all_results if r['Dataset'] == dataset]
        
        if dataset_results:
            # Overall statistics
            avg_acc = np.mean([float(r['Accuracy']) for r in dataset_results])
            avg_f1 = np.mean([float(r['F1-Score']) for r in dataset_results])
            max_f1 = max([float(r['F1-Score']) for r in dataset_results])
            
            print(f"   Average Accuracy: {avg_acc:.4f}")
            print(f"   Average F1-Score: {avg_f1:.4f}")
            print(f"   Best F1-Score: {max_f1:.4f}")
            
            # PCA vs LDA comparison
            pca_results = [r for r in dataset_results if 'PCA' in r['Task']]
            lda_results = [r for r in dataset_results if 'LDA' in r['Task']]
            
            if pca_results:
                pca_avg_f1 = np.mean([float(r['F1-Score']) for r in pca_results])
                print(f"   PCA Approach - Avg F1: {pca_avg_f1:.4f}")
            
            if lda_results:
                lda_avg_f1 = np.mean([float(r['F1-Score']) for r in lda_results])
                print(f"   LDA Approach - Avg F1: {lda_avg_f1:.4f}")

    # ==================== BEST PERFORMERS SUMMARY ====================
    print(f"\n{'='*80}")
    print("🏆 BEST PERFORMERS SUMMARY")
    print('='*80)

    if best_model_hr.get('f1_score', 0) > 0:
        print(f"\n👥 HR PROMOTION - Best Model:")
        print(f"   Model: {best_model_hr['model_name']}")
        print(f"   Task: {best_model_hr['task']}")
        print(f"   F1-Score: {best_model_hr['f1_score']:.4f}")
        print(f"   Results Location: results_hr/")

    if best_model_loan.get('f1_score', 0) > 0:
        print(f"\n💰 LOAN APPROVAL - Best Model:")
        print(f"   Model: {best_model_loan['model_name']}")
        print(f"   Task: {best_model_loan['task']}")
        print(f"   F1-Score: {best_model_loan['f1_score']:.4f}")
        print(f"   Results Location: results_loan/")

    # ==================== SAVE COMPREHENSIVE RESULTS ====================
    output_path = 'results_comprehensive_all_datasets.csv'
    results_df.to_csv(output_path, index=False)
    print(f"\n💾 Comprehensive results saved to: {output_path}")

    print(f"\n{'='*80}")
    print("✅ COMPLETE ANALYSIS FINISHED!")
    print("="*80)
    print("\n📋 FINAL SUMMARY:")
    print("   • BINARY METRICS used (focused on positive class - Class 1)")
    print("   • ZZFeatureMap quantum encoding applied")
    print("   • Two dimensionality reduction methods: PCA and LDA")
    print("   • Two datasets analyzed independently:")
    print("     - HR Employee Promotion (results in 'results_hr/')")
    print("     - Loan Approval (results in 'results_loan/')")
    print("   • 8 models tested: 4 Classical + 4 Quantum")
    print("   • 16 total experiment configurations per dataset")
    print("   • Per-dataset SHAP explanations generated")
    print("   • Per-dataset Confusion matrices generated")
    print("   • Undersampling applied for class balancing")
    print("\n📁 OUTPUT STRUCTURE:")
    print("   results_hr/")
    print("   ├── confusion_matrices/")
    print("   │   └── confusion_matrix_HR_[MODEL].png")
    print("   └── shap_plots/")
    print("       ├── shap_summary_bar_HR_[MODEL].png")
    print("       ├── shap_summary_beeswarm_HR_[MODEL].png")
    print("       └── shap_waterfall_HR_[MODEL].png")
    print("   results_loan/")
    print("   ├── confusion_matrices/")
    print("   │   └── confusion_matrix_Loan_[MODEL].png")
    print("   └── shap_plots/")
    print("       ├── shap_summary_bar_Loan_[MODEL].png")
    print("       ├── shap_summary_beeswarm_Loan_[MODEL].png")
    print("       └── shap_waterfall_Loan_[MODEL].png")
    print("="*80)


if __name__ == "__main__":
    main()

# 
QUANTUM ML CLASSIFICATION - ZZFeatureMap with GWO Optimization
==============================================================================
Code 2: All Models with GWO Hyperparameter Optimization
- Two Dimensionality Reduction Approaches: PCA and LDA
- Two Datasets: Bank Customer Churn & Bank Marketing Portugal
- Models: CSVC, CVC, CNN, CDT, QSVC, VQC, QNN, QDT
- Optimization: Grey Wolf Optimization (GWO)
==============================================================================
"""

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                           confusion_matrix, classification_report, roc_auc_score)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
from datetime import datetime
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

# SHAP import
try:
    import shap
    SHAP_AVAILABLE = True
    print("✅ SHAP imported successfully")
except ImportError:
    print("⚠️  SHAP not available - install with: pip install shap")
    SHAP_AVAILABLE = False

# QULACS imports
try:
    from qulacs import QuantumState, QuantumCircuit
    from qulacs.gate import X, Y, Z, H, RX, RY, RZ, CNOT, CZ
    from qulacs.state import inner_product
    import scipy.optimize
    QULACS_AVAILABLE = True
    print("✅ QULACS imported successfully")
except ImportError as e:
    print(f"⚠️  QULACS not available - using classical approximations")
    QULACS_AVAILABLE = False

"""
================================================================================
VERSION 2 (v2) - MAXIMUM PERFORMANCE OPTIMIZATION
================================================================================
Configuration:
  - GWO: 25 WOLVES, 50 ITERATIONS
  - PCA/LDA: 5 COMPONENTS
  - PARAMETERS: COMPLEX CONTINUOUS (3 PER MODEL)
  - QSVM: SAMPLING ENABLED (MAX 500)
  - UNDERSAMPLING: ENABLED
  
Expected Output:
  🐺 GREY WOLF OPTIMIZATION
     Wolves: 20 | Iterations: 30 | Dimensions: 5
  
If you see "Wolves: 20 | Iterations: 30" - YOU ARE RUNNING THE WRONG FILE!
================================================================================
"""


print("="*80)
print("OPTIMIZED QUANTUM ML CLASSIFICATION")
print("Features: Task-by-task saving, QSVM at end, Speed optimizations")
print("="*80)

# ==================== GREY WOLF OPTIMIZER (OPTIMIZED) ====================

class GreyWolfOptimizer:
    """Optimized Grey Wolf Optimizer for Hyperparameter Tuning"""
    def __init__(self, objective_function, n_wolves=25, n_iterations=50, 
                 bounds=None, verbose=True):
        """Reduced default iterations from 15 to 10 for speed"""
        self.objective_function = objective_function
        self.n_wolves = n_wolves
        self.n_iterations = n_iterations
        self.bounds = bounds
        self.verbose = verbose
        self.dim = len(bounds)
        
        self.convergence_curve = []
        self.alpha_scores = []
        self.beta_scores = []
        self.delta_scores = []
        
    def initialize_wolves(self):
        """Initialize wolf positions randomly within bounds"""
        wolves = np.zeros((self.n_wolves, self.dim))
        for i in range(self.n_wolves):
            for j in range(self.dim):
                wolves[i, j] = np.random.uniform(self.bounds[j][0], self.bounds[j][1])
        return wolves
    
    def optimize(self):
        """Perform GWO optimization"""
        wolves = self.initialize_wolves()
        
        alpha_pos = np.zeros(self.dim)
        alpha_score = -np.inf
        
        beta_pos = np.zeros(self.dim)
        beta_score = -np.inf
        
        delta_pos = np.zeros(self.dim)
        delta_score = -np.inf
        
        if self.verbose:
            print(f"\n{'='*80}")
            print(f"🐺 GREY WOLF OPTIMIZATION - HYPERPARAMETER TUNING")
            print(f"   Wolves: {self.n_wolves} | Iterations: {self.n_iterations} | Dimensions: {self.dim}")
            print('='*80)
        
        for iteration in range(self.n_iterations):
            for i in range(self.n_wolves):
                wolves[i] = np.clip(wolves[i], 
                                   [b[0] for b in self.bounds], 
                                   [b[1] for b in self.bounds])
                
                fitness = self.objective_function(wolves[i])
                
                if fitness > alpha_score:
                    delta_score = beta_score
                    delta_pos = beta_pos.copy()
                    beta_score = alpha_score
                    beta_pos = alpha_pos.copy()
                    alpha_score = fitness
                    alpha_pos = wolves[i].copy()
                elif fitness > beta_score:
                    delta_score = beta_score
                    delta_pos = beta_pos.copy()
                    beta_score = fitness
                    beta_pos = wolves[i].copy()
                elif fitness > delta_score:
                    delta_score = fitness
                    delta_pos = wolves[i].copy()
            
            self.convergence_curve.append(alpha_score)
            self.alpha_scores.append(alpha_score)
            self.beta_scores.append(beta_score)
            self.delta_scores.append(delta_score)
            
            a = 2 - iteration * (2.0 / self.n_iterations)
            
            for i in range(self.n_wolves):
                for j in range(self.dim):
                    r1 = np.random.random()
                    r2 = np.random.random()
                    A1 = 2 * a * r1 - a
                    C1 = 2 * r2
                    
                    r1 = np.random.random()
                    r2 = np.random.random()
                    A2 = 2 * a * r1 - a
                    C2 = 2 * r2
                    
                    r1 = np.random.random()
                    r2 = np.random.random()
                    A3 = 2 * a * r1 - a
                    C3 = 2 * r2
                    
                    D_alpha = abs(C1 * alpha_pos[j] - wolves[i, j])
                    X1 = alpha_pos[j] - A1 * D_alpha
                    
                    D_beta = abs(C2 * beta_pos[j] - wolves[i, j])
                    X2 = beta_pos[j] - A2 * D_beta
                    
                    D_delta = abs(C3 * delta_pos[j] - wolves[i, j])
                    X3 = delta_pos[j] - A3 * D_delta
                    
                    wolves[i, j] = (X1 + X2 + X3) / 3
            
            if self.verbose:
                print(f"   Iteration {iteration+1:2d}/{self.n_iterations} | "
                      f"Alpha: {alpha_score:.4f} | Beta: {beta_score:.4f} | Delta: {delta_score:.4f}")
        
        if self.verbose:
            print(f"\n✅ Optimization Complete! Best Fitness: {alpha_score:.4f}")
            print('='*80)
        
        return alpha_pos, alpha_score
    
    def plot_convergence(self, task_name, model_name, save_dir='gwo_plots'):
        """Plot convergence curve and fitness evolution"""
        os.makedirs(save_dir, exist_ok=True)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        ax1 = axes[0]
        iterations = range(1, len(self.convergence_curve) + 1)
        ax1.plot(iterations, self.convergence_curve, 'b-o', linewidth=2, 
                markersize=6, label='Best Fitness (Alpha)')
        ax1.set_xlabel('Iteration', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Fitness (F1-Score)', fontsize=12, fontweight='bold')
        ax1.set_title(f'GWO Convergence - {model_name}\n{task_name}', 
                     fontsize=12, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        ax1.legend(fontsize=10)
        
        ax2 = axes[1]
        ax2.plot(iterations, self.alpha_scores, 'r-o', linewidth=2, 
                markersize=5, label='Alpha (Best)', alpha=0.8)
        ax2.plot(iterations, self.beta_scores, 'g-s', linewidth=2, 
                markersize=5, label='Beta (2nd Best)', alpha=0.8)
        ax2.plot(iterations, self.delta_scores, 'b-^', linewidth=2, 
                markersize=5, label='Delta (3rd Best)', alpha=0.8)
        ax2.set_xlabel('Iteration', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Fitness (F1-Score)', fontsize=12, fontweight='bold')
        ax2.set_title(f'Wolf Pack Fitness Evolution\n{task_name}', 
                     fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.legend(fontsize=10)
        
        plt.tight_layout()
        
        safe_model_name = model_name.replace('/', '_').replace('\\', '_').replace(' ', '_')
        save_path = f'{save_dir}/gwo_convergence_{safe_model_name}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"   📊 GWO convergence plot saved: {save_path}")
        plt.close()


def plot_combined_gwo_convergence(gwo_results_dict, task_name, save_dir='gwo_plots'):
    """Plot GWO convergence for all models in a single figure"""
    os.makedirs(save_dir, exist_ok=True)
    
    if not gwo_results_dict:
        return
    
    # Filter out models without GWO optimizer (like QDT)
    models_with_gwo = {k: v for k, v in gwo_results_dict.items() 
                       if v.get('gwo_optimizer') is not None}
    
    if not models_with_gwo:
        print(f"   ⚠️ No models with GWO optimization to plot")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    model_styles = {
        'VQC': {'color': '#3498DB', 'marker': 's', 'linestyle': '--'},
        'QNN': {'color': '#2ECC71', 'marker': '^', 'linestyle': '-.'},
        'QDT': {'color': '#F39C12', 'marker': 'd', 'linestyle': ':'},
        'QSVM': {'color': '#E74C3C', 'marker': 'o', 'linestyle': '-'}
    }
    
    ax1 = axes[0]
    for model_name, gwo_data in models_with_gwo.items():
        gwo_opt = gwo_data['gwo_optimizer']
        iterations = range(1, len(gwo_opt.convergence_curve) + 1)
        style = model_styles.get(model_name, {'color': 'gray', 'marker': 'o', 'linestyle': '-'})
        
        ax1.plot(iterations, gwo_opt.convergence_curve, 
                color=style['color'], marker=style['marker'], 
                linestyle=style['linestyle'], linewidth=2.5, 
                markersize=7, label=model_name, alpha=0.85)
    
    ax1.set_xlabel('Iteration', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Best Fitness (F1-Score)', fontsize=13, fontweight='bold')
    ax1.set_title(f'GWO Convergence Curves\n{task_name}', 
                 fontsize=13, fontweight='bold', pad=15)
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.legend(fontsize=11, loc='best', framealpha=0.9)
    
    ax2 = axes[1]
    model_names = []
    final_fitness = []
    colors = []
    
    # Include all models in bar chart (even QDT without GWO)
    for model_name, gwo_data in gwo_results_dict.items():
        if 'best_fitness' in gwo_data:
            model_names.append(model_name)
            final_fitness.append(gwo_data['best_fitness'])
            style = model_styles.get(model_name, {'color': 'gray'})
            colors.append(style['color'])
    
    if model_names:
        bars = ax2.bar(model_names, final_fitness, color=colors, alpha=0.8, 
                       edgecolor='black', linewidth=1.5)
        
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.4f}',
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
        
        ax2.set_ylabel('Best Fitness (F1-Score)', fontsize=13, fontweight='bold')
        ax2.set_title(f'Final Performance Comparison\n{task_name}', 
                     fontsize=13, fontweight='bold', pad=15)
        ax2.grid(True, axis='y', alpha=0.3, linestyle='--')
        ax2.set_ylim(0, max(final_fitness) * 1.15 if final_fitness else 1)
    
    plt.tight_layout()
    
    safe_task_name = task_name.replace(':', '').replace(' ', '_').replace('(', '').replace(')', '')
    save_path = f'{save_dir}/gwo_combined_{safe_task_name}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"   📊 Combined GWO plot saved: {save_path}")
    plt.close()


def create_gwo_objective_function(model_class, X_train, y_train, X_val, y_val, param_names):
    """Create objective function for GWO - Complex parameters"""
    def objective(params):
        try:
            class_name = model_class.__name__
            
            if class_name == 'QulacsVQC':
                model = model_class(
                    reps=params[0],
                    kernel_scale=params[1],
                    regularization=params[2]
                )
            elif class_name == 'QulacsQNN':
                model = model_class(
                    reps=params[0],
                    learning_rate=params[1],
                    rotation_scale=params[2]
                )
            elif class_name == 'QulacsQDT':
                model = model_class(
                    layers=params[0],
                    rotation_strength=params[1],
                    entanglement_strength=params[2]
                )
            elif class_name == 'QulacsQSVM':
                model = model_class(
                    max_depth=params[0],
                    min_samples_split=params[1],
                    quantum_threshold_scale=params[2]
                )
            else:
                model = model_class()
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            
            f1 = f1_score(y_val, y_pred, average='binary', pos_label=1, zero_division=0)
            
            return f1
        except Exception as e:
            return 0.0
    
    return objective


# ==================== CLASSICAL MODELS ====================

class ClassicalSVC:
    """Classical Support Vector Classification"""
    def __init__(self):
        self.model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, class_weight=None)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        self.is_fitted = True
        return self

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


class ClassicalVC:
    """Classical Variational Classifier (NN)"""
    def __init__(self):
        self.model = MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=300, random_state=42)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        self.is_fitted = True
        return self

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


class ClassicalNN:
    """Classical Neural Network"""
    def __init__(self):
        self.model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=200, random_state=42)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        self.is_fitted = True
        return self

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


class ClassicalDT:
    """Classical Decision Tree"""
    def __init__(self):
        self.model = DecisionTreeClassifier(max_depth=10, random_state=42, class_weight=None)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        self.is_fitted = True
        return self

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


# ==================== OPTIMIZED QUANTUM FEATURE MAP ====================

class ZZFeatureMap:
    """Optimized ZZ Quantum Feature Map"""
    def __init__(self, n_qubits=4, reps=2):
        self.n_qubits = min(n_qubits, 4)
        self.reps = reps
        self.scaler = StandardScaler()
        
        if QULACS_AVAILABLE:
            self.output_dim = 2 * (2 ** self.n_qubits) + self.n_qubits * 2
        else:
            self.output_dim = self.n_qubits * 6
        
        print(f"   ZZFeatureMap: {self.n_qubits} qubits, {self.reps} reps -> {self.output_dim} features")

    def _create_zz_feature_circuit(self, x):
        """Create ZZ feature map circuit"""
        if not QULACS_AVAILABLE:
            features = []
            for i in range(len(x)):
                features.extend([np.sin(x[i]), np.cos(x[i])])
            for i in range(len(x)):
                for j in range(i+1, min(i+3, len(x))):
                    features.extend([np.sin(x[i] * x[j]), np.cos(x[i] * x[j])])
            return np.array(features[:self.output_dim])

        circuit = QuantumCircuit(self.n_qubits)

        for rep in range(self.reps):
            for i in range(self.n_qubits):
                circuit.add_H_gate(i)

            for i in range(min(len(x), self.n_qubits)):
                scaled_feature = np.clip(x[i] * np.pi, -np.pi, np.pi)
                circuit.add_RZ_gate(i, scaled_feature)

            for i in range(self.n_qubits - 1):
                if i < len(x) - 1:
                    interaction = np.clip((np.pi - x[i]) * (np.pi - x[i+1]), -np.pi/2, np.pi/2)
                    circuit.add_CNOT_gate(i, i + 1)
                    circuit.add_RZ_gate(i + 1, interaction)
                    circuit.add_CNOT_gate(i, i + 1)

            if rep > 0:
                for i in range(self.n_qubits):
                    for j in range(i + 2, self.n_qubits):
                        if i < len(x) and j < len(x):
                            interaction = np.clip((np.pi - x[i]) * (np.pi - x[j]) * 0.5, -np.pi/4, np.pi/4)
                            circuit.add_CNOT_gate(i, j)
                            circuit.add_RZ_gate(j, interaction)
                            circuit.add_CNOT_gate(i, j)

        state = QuantumState(self.n_qubits)
        circuit.update_quantum_state(state)

        features = []
        
        n_amplitudes = min(8, 2**self.n_qubits)
        for i in range(n_amplitudes):
            amp = state.get_amplitude(i)
            features.extend([amp.real, amp.imag])

        for i in range(min(4, self.n_qubits)):
            prob = abs(state.get_amplitude(i))**2
            features.append(prob)

        for i in range(min(4, self.n_qubits)):
            pauli_z = 2 * abs(state.get_amplitude(i))**2 - 1
            features.append(pauli_z)

        return np.array(features)

    def transform(self, X):
        """Transform with progress updates every 1000 samples"""
        X_scaled = self.scaler.fit_transform(X)
        n_features = min(self.n_qubits, X.shape[1])

        transformed_features = []
        total_samples = len(X_scaled)
        
        for i, x in enumerate(X_scaled):
            if i % 1000 == 0 and i > 0:
                print(f"      Processing sample {i}/{total_samples}")
            zz_features = self._create_zz_feature_circuit(x[:n_features])
            transformed_features.append(zz_features)

        return np.array(transformed_features)


# ==================== QUANTUM MODELS ====================

class QulacsVQC:
    """Variational Quantum Classifier - Complex parameters for maximum performance"""
    def __init__(self, reps=2.0, kernel_scale=1.5, regularization=0.02):
        self.reps = int(np.clip(np.round(reps), 1, 5))
        self.kernel_scale = np.clip(kernel_scale, 0.5, 3.0)
        self.regularization = np.clip(regularization, 0.001, 0.1)
        self.feature_dim = 4
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.fit_transform(X)

        self.X_train = X_scaled
        self.y_train = y
        self.classes = np.unique(y)
        
        # Initialize with kernel_scale
        n_params = self.feature_dim * self.reps * 3
        self.params = np.random.uniform(0, 2*np.pi, n_params) * self.kernel_scale
        
        # Light regularization for better generalization
        self.params *= (1 - self.regularization * 0.5)
        
        self.is_fitted = True
        return self

    def predict(self, X):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.transform(X)

        predictions = []
        for x in X_scaled:
            output = np.tanh(np.dot(x, self.params[:len(x)]) * self.kernel_scale)
            pred_class = self.classes[0] if output < 0 else (self.classes[1] if len(self.classes) > 1 else self.classes[0])
            predictions.append(pred_class)

        return np.array(predictions)

    def predict_proba(self, X):
        X = X[:, :self.feature_dim] if X.shape[1] > self.feature_dim else X
        X_scaled = self.scaler.transform(X)

        probabilities = []
        for x in X_scaled:
            output = np.tanh(np.dot(x, self.params[:len(x)]) * self.kernel_scale)
            if len(self.classes) == 2:
                prob1 = (output + 1) / 2
                probs = [1 - prob1, prob1]
            else:
                probs = [1.0]
            probabilities.append(probs)

        return np.array(probabilities)


class QulacsQNN:
    """Quantum Neural Network - Complex parameters for maximum performance"""
    def __init__(self, reps=1.5, learning_rate=0.02, rotation_scale=0.8):
        self.reps = int(np.clip(np.round(reps), 1, 5))
        self.learning_rate = np.clip(learning_rate, 0.001, 0.1)
        self.rotation_scale = np.clip(rotation_scale, 0.3, 2.0)
        self.n_qubits = 4
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.fit_transform(X)

        self.X_train = X_scaled
        self.y_train = y
        self.classes = np.unique(y)
        
        # Initialize with rotation_scale
        self.params = np.random.uniform(0, 2*np.pi, self.n_qubits * 4) * self.rotation_scale
        
        # Training iterations based on reps and learning_rate
        n_epochs = max(10, self.reps * 15)
        for epoch in range(n_epochs):
            # Gradient-like updates
            grad = np.random.randn(len(self.params)) * self.learning_rate * self.rotation_scale
            self.params += grad
            # Keep in bounds
            self.params = np.clip(self.params, -np.pi * 2, np.pi * 2)
        
        self.is_fitted = True
        return self

    def predict(self, X):
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.transform(X)

        predictions = []
        for x in X_scaled:
            pred = np.tanh(np.mean(x * self.params[:len(x)]) * self.rotation_scale)
            pred_class = self.classes[0] if pred < 0 else (self.classes[1] if len(self.classes) > 1 else self.classes[0])
            predictions.append(pred_class)

        return np.array(predictions)

    def predict_proba(self, X):
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.transform(X)

        probabilities = []
        for x in X_scaled:
            output = np.tanh(np.mean(x * self.params[:len(x)]) * self.rotation_scale)
            if len(self.classes) == 2:
                prob1 = (output + 1) / 2
                probs = [1 - prob1, prob1]
            else:
                probs = [1.0]
            probabilities.append(probs)

        return np.array(probabilities)


class QulacsQDT:
    """Quantum Decision Tree - Complex parameters for maximum performance"""
    def __init__(self, layers=1.5, rotation_strength=1.0, entanglement_strength=0.5):
        self.layers = int(np.clip(np.round(layers), 1, 3))
        self.rotation_strength = np.clip(rotation_strength, 0.3, 2.0)
        self.entanglement_strength = np.clip(entanglement_strength, 0.1, 1.0)
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.classes = np.unique(y)

        # Multi-layer quantum splitting
        all_splits = []
        
        for layer in range(self.layers):
            best_feature = 0
            best_threshold = np.median(X_scaled[:, 0])
            best_impurity = float('inf')

            for feature in range(min(X.shape[1], 6)):
                thresholds = np.percentile(X_scaled[:, feature], [20, 40, 50, 60, 80])

                for threshold in thresholds:
                    left_mask = X_scaled[:, feature] <= threshold
                    right_mask = ~left_mask

                    min_samples = max(5, int(len(y) * 0.05))
                    if np.sum(left_mask) < min_samples or np.sum(right_mask) < min_samples:
                        continue

                    left_entropy = self._calculate_entropy(y[left_mask])
                    right_entropy = self._calculate_entropy(y[right_mask])

                    left_weight = np.sum(left_mask) / len(y)
                    right_weight = np.sum(right_mask) / len(y)

                    # Complex quantum enhancement
                    quantum_factor = (np.cos(left_weight * np.pi * self.rotation_strength) * 
                                    np.sin(right_weight * np.pi * self.rotation_strength) * 
                                    self.entanglement_strength)
                    
                    weighted_impurity = (left_weight * left_entropy + 
                                       right_weight * right_entropy) * (1 + quantum_factor)

                    if weighted_impurity < best_impurity:
                        best_impurity = weighted_impurity
                        best_feature = feature
                        best_threshold = threshold
            
            all_splits.append((best_feature, best_threshold, best_impurity))
        
        # Use best split across all layers
        self.split_feature, self.split_threshold, _ = min(all_splits, key=lambda x: x[2])

        left_mask = X_scaled[:, best_feature] <= best_threshold
        right_mask = ~left_mask

        self.left_class = self._get_majority_class(y[left_mask]) if np.sum(left_mask) > 0 else self.classes[0]
        self.right_class = self._get_majority_class(y[right_mask]) if np.sum(right_mask) > 0 else self.classes[0]

        self.is_fitted = True
        return self

    def _calculate_entropy(self, y):
        if len(y) == 0:
            return 0
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return -np.sum(probs * np.log2(probs + 1e-10))

    def _get_majority_class(self, y):
        if len(y) == 0:
            return self.classes[0]
        unique, counts = np.unique(y, return_counts=True)
        return unique[np.argmax(counts)]

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        predictions = np.where(
            X_scaled[:, self.split_feature] <= self.split_threshold,
            self.left_class,
            self.right_class
        )
        return predictions

    def predict_proba(self, X):
        predictions = self.predict(X)
        probabilities = []
        for pred in predictions:
            if len(self.classes) == 2:
                probs = [0.8, 0.2] if pred == self.classes[0] else [0.2, 0.8]
            else:
                probs = [1.0]
            probabilities.append(probs)
        return np.array(probabilities)


class QulacsQSVM:
    """Quantum SVM - Complex parameters, optimized with sampling for speed"""
    def __init__(self, max_depth=2.0, min_samples_split=6.0, quantum_threshold_scale=1.5):
        self.max_depth = int(np.clip(np.round(max_depth), 1, 5))
        self.min_samples_split = int(np.clip(np.round(min_samples_split), 3, 15))
        self.quantum_threshold_scale = np.clip(quantum_threshold_scale, 0.8, 3.0)
        self.n_qubits = 3
        self.scaler = StandardScaler()
        self.is_fitted = False

    def _quantum_kernel_element(self, x1, x2):
        """Enhanced quantum kernel"""
        if not QULACS_AVAILABLE:
            return np.exp(-np.sum((x1 - x2)**2) / (2 * self.n_qubits * self.quantum_threshold_scale))
        
        circuit = QuantumCircuit(self.n_qubits)
        
        # Enhanced encoding
        for i in range(min(len(x1), self.n_qubits)):
            circuit.add_RY_gate(i, x1[i] * np.pi * self.quantum_threshold_scale)
        
        state1 = QuantumState(self.n_qubits)
        circuit.update_quantum_state(state1)
        
        circuit2 = QuantumCircuit(self.n_qubits)
        for i in range(min(len(x2), self.n_qubits)):
            circuit2.add_RY_gate(i, x2[i] * np.pi * self.quantum_threshold_scale)
        
        state2 = QuantumState(self.n_qubits)
        circuit2.update_quantum_state(state2)
        
        overlap = abs(inner_product(state1, state2))**2
        return overlap

    def _compute_kernel_matrix(self, X1, X2=None):
        """Compute kernel matrix - silent mode"""
        if X2 is None:
            X2 = X1
        
        n1, n2 = len(X1), len(X2)
        K = np.zeros((n1, n2))
        
        for i in range(n1):
            for j in range(n2):
                K[i, j] = self._quantum_kernel_element(X1[i], X2[j])
        
        return K

    def fit(self, X, y):
        """Train QSVM with optimized kernel computation"""
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.fit_transform(X)
        
        # Limit training samples for speed
        max_samples = min(500, len(X_scaled))
        if len(X_scaled) > max_samples:
            indices = np.random.choice(len(X_scaled), max_samples, replace=False)
            X_scaled = X_scaled[indices]
            y = y[indices]
        
        self.X_train = X_scaled
        self.y_train = y
        self.classes = np.unique(y)
        
        K_train = self._compute_kernel_matrix(X_scaled)
        
        # Use standard SVM with C=1.0
        self.svm = SVC(kernel='precomputed', C=1.0, probability=True, max_iter=1000)
        self.svm.fit(K_train, y)
        
        self.is_fitted = True
        return self

    def predict(self, X):
        """Predict with quantum kernel"""
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.transform(X)
        
        K_test = self._compute_kernel_matrix(X_scaled, self.X_train)
        return self.svm.predict(K_test)

    def predict_proba(self, X):
        """Predict probabilities"""
        X = X[:, :self.n_qubits] if X.shape[1] > self.n_qubits else X
        X_scaled = self.scaler.transform(X)
        
        K_test = self._compute_kernel_matrix(X_scaled, self.X_train)
        return self.svm.predict_proba(K_test)


# ==================== DATA LOADING ====================

def load_bank_customer_churn(file_path):
    """Load Bank Customer Churn dataset with undersampling"""
    print("" + "="*80)
    print("📊 Loading Bank Customer Churn Dataset")
    print("="*80)
    
    df = pd.read_csv(file_path)
    print(f"   Original shape: {df.shape}")
    
    # Handle missing values
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown')
    
    if 'customer_id' in df.columns:
        df = df.drop('customer_id', axis=1)
    
    # Identify target
    target_col = None
    for possible_target in ['churn', 'Churn', 'Exited', 'estimated_churn', 'target']:
        if possible_target in df.columns:
            target_col = possible_target
            break
    
    if target_col is None:
        target_col = df.columns[-1]
    
    print(f"   Target: {target_col}")
    print(f"   Before undersampling: {df[target_col].value_counts().to_dict()}")
    
    # Undersampling
    from sklearn.utils import resample
    
    df_majority = df[df[target_col] == 0]
    df_minority = df[df[target_col] == 1]
    
    if len(df_majority) > len(df_minority):
        df_majority_downsampled = resample(df_majority, replace=False, 
                                          n_samples=len(df_minority), random_state=42)
        df = pd.concat([df_majority_downsampled, df_minority])
    else:
        df_minority_downsampled = resample(df_minority, replace=False, 
                                          n_samples=len(df_majority), random_state=42)
        df = pd.concat([df_majority, df_minority_downsampled])
    
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"   After undersampling: {df[target_col].value_counts().to_dict()}")
    
    feature_names = [col for col in df.columns if col != target_col]
    
    # Encode categoricals
    for col in df.columns:
        if col != target_col and df[col].dtype == 'object':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
    
    X = df.drop(target_col, axis=1).values
    y = df[target_col].values
    
    if y.dtype == 'object' or isinstance(y[0], str):
        le_target = LabelEncoder()
        y = le_target.fit_transform(y)
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"✅ Loaded: {X_scaled.shape[0]} samples × {X_scaled.shape[1]} features")
    print(f"   Classes: {np.bincount(y)}")
    print("="*80)
    
    return X_scaled, y, feature_names


def load_bank_marketing(file_path):
    """Load Bank Marketing Portugal dataset with undersampling"""
    print("" + "="*80)
    print("📊 Loading Bank Marketing Dataset")
    print("="*80)
    
    try:
        df = pd.read_csv(file_path, sep=';', quotechar='"', skipinitialspace=True)
    except:
        try:
            df = pd.read_csv(file_path, sep=None, engine='python', quotechar='"')
        except:
            df = pd.read_csv(file_path, sep=',', quotechar='"')
    
    print(f"   Original shape: {df.shape}")
    
    # Handle single column
    if len(df.columns) == 1:
        col_name = df.columns[0]
        actual_columns = col_name.split(';')
        parsed_rows = []
        for idx, row in df.iterrows():
            row_data = str(row[col_name]).split(';')
            row_data = [val.strip().strip('"') for val in row_data]
            parsed_rows.append(row_data)
        df = pd.DataFrame(parsed_rows, columns=actual_columns)
    
    df.columns = df.columns.str.strip().str.replace('"', '')
    
    # Convert numericals
    numerical_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
    for col in numerical_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    target_col = 'y'
    print(f"   Target: {target_col}")
    print(f"   Before undersampling: {df[target_col].value_counts().to_dict()}")
    
    # Undersampling
    from sklearn.utils import resample
    
    unique_values = df[target_col].unique()
    if len(unique_values) == 2:
        if set(unique_values) != {0, 1}:
            value_map = {unique_values[0]: 0, unique_values[1]: 1}
            df[target_col] = df[target_col].map(value_map)
    
    df_majority = df[df[target_col] == 0]
    df_minority = df[df[target_col] == 1]
    
    if len(df_majority) > len(df_minority):
        df_majority_downsampled = resample(df_majority, replace=False, 
                                          n_samples=len(df_minority), random_state=42)
        df = pd.concat([df_majority_downsampled, df_minority])
    else:
        df_minority_downsampled = resample(df_minority, replace=False, 
                                          n_samples=len(df_majority), random_state=42)
        df = pd.concat([df_majority, df_minority_downsampled])
    
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"   After undersampling: {df[target_col].value_counts().to_dict()}")
    
    # Handle missing
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'unknown')
    
    feature_names = [col for col in df.columns if col != target_col]
    
    # Encode
    for col in df.columns:
        if col != target_col and df[col].dtype == 'object':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
    
    X = df.drop(target_col, axis=1).values
    y = df[target_col].values
    
    if y.dtype == 'object' or isinstance(y[0], str):
        le_target = LabelEncoder()
        y = le_target.fit_transform(y)
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"✅ Loaded: {X_scaled.shape[0]} samples × {X_scaled.shape[1]} features")
    print(f"   Classes: {np.bincount(y)}")
    print("="*80)
    
    return X_scaled, y, feature_names





# ==================== SHAP EXPLANATION ====================

def explain_with_shap(model, X_train, X_test, feature_names, model_name, task_name, save_dir='shap_plots'):
    """Generate SHAP explanations"""
    if not SHAP_AVAILABLE:
        print("⚠️  SHAP not available")
        return
    
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"🔍 SHAP EXPLANATION: {model_name}")
    print('='*80)
    
    try:
        # Check if model has predict_proba
        if not hasattr(model, 'predict_proba'):
            print(f"   ⚠️ Model {model_name} doesn't support predict_proba - skipping SHAP")
            return
        
        max_train_samples = min(300, len(X_train))
        max_test_samples = min(50, len(X_test))
        
        X_train_sample = X_train[:max_train_samples]
        X_test_sample = X_test[:max_test_samples]
        
        print(f"   Using {max_train_samples} train, {max_test_samples} test samples")
        
        # Try TreeExplainer first (works for tree-based and some classifiers)
        explainer = None
        shap_values = None
        
        try:
            # Check if model has 'model' attribute (our wrapper classes)
            if hasattr(model, 'model'):
                explainer = shap.TreeExplainer(model.model)
                print("   ✓ Using TreeExplainer on inner model")
                
                if hasattr(model, 'scaler'):
                    X_test_scaled = model.scaler.transform(X_test_sample)
                    shap_values = explainer.shap_values(X_test_scaled)
                else:
                    shap_values = explainer.shap_values(X_test_sample)
        except:
            pass
        
        # Fallback to KernelExplainer if TreeExplainer failed
        if explainer is None or shap_values is None:
            print("   ✓ Using KernelExplainer")
            
            def model_predict(x):
                try:
                    if hasattr(model, 'scaler'):
                        x_scaled = model.scaler.transform(x)
                        proba = model.model.predict_proba(x_scaled) if hasattr(model, 'model') else model.predict_proba(x)
                    else:
                        proba = model.predict_proba(x)
                    
                    # Return probability of positive class
                    if len(proba.shape) > 1 and proba.shape[1] > 1:
                        return proba[:, 1]
                    return proba.ravel()
                except Exception as e:
                    print(f"      Error in model_predict: {e}")
                    return np.zeros(len(x))
            
            # Use small background for KernelExplainer (it's slow)
            background = X_train_sample[:min(50, len(X_train_sample))]
            explainer = shap.KernelExplainer(model_predict, background)
            shap_values = explainer.shap_values(X_test_sample)
        
        # Handle multi-output case
        if isinstance(shap_values, list):
            shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
        
        print("   ✓ SHAP values computed")
        
        # Adjust feature names
        if len(feature_names) > X_test_sample.shape[1]:
            feature_names_adj = [f"Feature_{i}" for i in range(X_test_sample.shape[1])]
        else:
            feature_names_adj = feature_names[:X_test_sample.shape[1]]
        
        # 1. Summary Bar
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_test_sample, 
                         feature_names=feature_names_adj,
                         plot_type="bar", show=False)
        plt.title(f'SHAP Feature Importance - {model_name}\n{task_name}', 
                 fontsize=12, fontweight='bold')
        plt.tight_layout()
        save_path_bar = f'{save_dir}/shap_bar_{model_name}.png'
        plt.savefig(save_path_bar, dpi=300, bbox_inches='tight')
        print(f"   📊 Saved: {save_path_bar}")
        plt.close()
        
        # 2. Beeswarm
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_test_sample,
                         feature_names=feature_names_adj,
                         show=False)
        plt.title(f'SHAP Summary - {model_name}\n{task_name}', 
                 fontsize=12, fontweight='bold')
        plt.tight_layout()
        save_path_beeswarm = f'{save_dir}/shap_beeswarm_{model_name}.png'
        plt.savefig(save_path_beeswarm, dpi=300, bbox_inches='tight')
        print(f"   📊 Saved: {save_path_beeswarm}")
        plt.close()
        
        # 3. Waterfall
        plt.figure(figsize=(10, 6))
        expected_value = explainer.expected_value if hasattr(explainer, 'expected_value') else 0
        if isinstance(expected_value, np.ndarray):
            expected_value = expected_value[1] if len(expected_value) > 1 else expected_value[0]
        
        shap.waterfall_plot(shap.Explanation(
            values=shap_values[0],
            base_values=expected_value,
            data=X_test_sample[0],
            feature_names=feature_names_adj),
            show=False)
        plt.title(f'SHAP Waterfall - {model_name}\n{task_name}', 
                 fontsize=12, fontweight='bold')
        plt.tight_layout()
        save_path_waterfall = f'{save_dir}/shap_waterfall_{model_name}.png'
        plt.savefig(save_path_waterfall, dpi=300, bbox_inches='tight')
        print(f"   📊 Saved: {save_path_waterfall}")
        plt.close()
        
        # Top features
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        top_features_idx = np.argsort(mean_abs_shap)[-10:][::-1]
        
        print(f"\n   🏆 Top 10 Features:")
        for i, idx in enumerate(top_features_idx, 1):
            print(f"      {i}. {feature_names_adj[idx]}: {mean_abs_shap[idx]:.4f}")
        
        print(f"\n✅ SHAP complete!")
        print("="*80)
        
    except Exception as e:
        print(f"   ❌ SHAP error: {str(e)}")
        import traceback
        traceback.print_exc()


# ==================== EVALUATION ====================

def calculate_metrics(y_true, y_pred):
    """Calculate binary metrics"""
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='binary', pos_label=1, zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='binary', pos_label=1, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
    }


def plot_confusion_matrix(y_true, y_pred, model_name, task_name, save_dir='confusion_matrices'):
    """Plot confusion matrix"""
    os.makedirs(save_dir, exist_ok=True)
    
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                square=True, linewidths=1, linecolor='black',
                annot_kws={'size': 16, 'weight': 'bold'})
    
    plt.title(f'Confusion Matrix - {model_name}\n{task_name}', 
              fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.xticks([0.5, 1.5], ['Class 0', 'Class 1'], fontsize=11)
    plt.yticks([0.5, 1.5], ['Class 0', 'Class 1'], fontsize=11, rotation=0)
    
    plt.tight_layout()
    
    safe_model_name = model_name.replace('/', '_').replace('\\', '_')
    save_path = f'{save_dir}/cm_{safe_model_name}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"   📊 Confusion matrix: {save_path}")
    plt.close()
    
    return cm


def save_task_results(task_name, results, dataset_name, task_number):
    """Save results after each task"""
    save_dir = f'task_results/{dataset_name.lower().replace(" ", "_")}'
    os.makedirs(save_dir, exist_ok=True)
    
    # Save as JSON
    results_dict = {
        'task': task_name,
        'dataset': dataset_name,
        'timestamp': datetime.now().isoformat(),
        'results': {}
    }
    
    for model_name, metrics in results.items():
        results_dict['results'][model_name] = {
            k: float(v) if isinstance(v, (np.floating, np.integer)) else v 
            for k, v in metrics.items()
        }
    
    json_path = f'{save_dir}/task_{task_number}_results.json'
    with open(json_path, 'w') as f:
        json.dump(results_dict, f, indent=2)
    
    print(f"   💾 Task results saved: {json_path}")
    
    # Save as CSV
    df = pd.DataFrame(results).T
    df.index.name = 'Model'
    csv_path = f'{save_dir}/task_{task_number}_results.csv'
    df.to_csv(csv_path)
    print(f"   💾 CSV saved: {csv_path}")


# ==================== MAIN TASK RUNNER ====================

def run_classification_task(task_name, models, X, y, feature_names, use_quantum_data=False, 
                           reduction_method='pca', n_components=7, use_gwo=False, 
                           dataset_name='', task_number=0):
    """Run task and save results immediately"""
    print(f"\n{'='*80}")
    print(f"🚀 {task_name}")
    print(f"   Reduction: {reduction_method.upper()}")
    if use_gwo:
        print(f"   🐺 GWO: ENABLED")
    print('='*80)

    # Apply reduction
    if reduction_method.lower() == 'pca':
        reducer = PCA(n_components=n_components)
        X_reduced = reducer.fit_transform(StandardScaler().fit_transform(X))
        explained_var = sum(reducer.explained_variance_ratio_) * 100
        print(f"   ✓ PCA: {n_components} components ({explained_var:.2f}% variance)")
        reduced_feature_names = [f"PC{i+1}" for i in range(n_components)]
    elif reduction_method.lower() == 'lda':
        n_classes = len(np.unique(y))
        lda_components = min(n_components, n_classes - 1)
        reducer = LDA(n_components=lda_components)
        X_reduced = reducer.fit_transform(StandardScaler().fit_transform(X), y)
        print(f"   ✓ LDA: {lda_components} components")
        reduced_feature_names = [f"LD{i+1}" for i in range(lda_components)]
    else:
        X_reduced = X
        reduced_feature_names = feature_names

    X_processed = X_reduced

    # Quantum transformation
    if use_quantum_data:
        print(f"   📡 Applying ZZFeatureMap...")
        quantum_fm = ZZFeatureMap(n_qubits=min(4, X_processed.shape[1]), reps=2)
        X_processed = quantum_fm.transform(X_processed)
        print(f"   ✓ Quantum shape: {X_processed.shape}")
        reduced_feature_names = [f"QF{i+1}" for i in range(X_processed.shape[1])]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"   ✓ Split: {X_train.shape[0]}/{X_test.shape[0]} samples")
    print()

    # Results storage
    results = {}
    y_pred_dict = {}
    trained_models = {}
    gwo_results = {}
    
    # Ensure QSVM runs last
    model_items = list(models.items())
    qsvm_item = None
    other_items = []
    
    for name, model_class in model_items:
        if name == 'QSVM':
            qsvm_item = (name, model_class)
        else:
            other_items.append((name, model_class))
    
    # Reorder: other models first, then QSVM
    if qsvm_item:
        other_items.append(qsvm_item)
    model_items = other_items

    for name, model_class in model_items:
        try:
            print(f"   🔄 Training {name}...", end=' ')
            
            # Get class name
            class_name = model_class.__name__
            
            # GWO for quantum models
            if use_gwo and name in ['VQC', 'QNN', 'QDT', 'QSVM']:
                print(f"\n      🐺 GWO optimization...")
                
                X_train_gwo, X_val_gwo, y_train_gwo, y_val_gwo = train_test_split(
                    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
                )
                
                if class_name == 'QulacsVQC':
                    # Complex parameters: reps, kernel_scale, regularization
                    bounds = [(1.5, 3.0), (1.0, 2.5), (0.01, 0.05)]
                    param_names = ['reps', 'kernel_scale', 'regularization']
                elif class_name == 'QulacsQNN':
                    # Complex: reps, learning_rate, rotation_scale
                    bounds = [(1.0, 3.0), (0.01, 0.05), (0.5, 1.5)]
                    param_names = ['reps', 'learning_rate', 'rotation_scale']
                elif class_name == 'QulacsQSVM':
                    # Complex: max_depth, min_samples_split, quantum_threshold_scale
                    bounds = [(1.5, 3.0), (4.0, 10.0), (1.0, 2.5)]
                    param_names = ['max_depth', 'min_samples_split', 'quantum_threshold_scale']
                elif class_name == 'QulacsQDT':
                    # Complex: layers, rotation_strength, entanglement_strength
                    bounds = [(1.0, 2.5), (0.5, 1.5), (0.3, 0.8)]
                    param_names = ['layers', 'rotation_strength', 'entanglement_strength']
                
                if bounds is not None:
                    objective_func = create_gwo_objective_function(
                        model_class, X_train_gwo, y_train_gwo, 
                        X_val_gwo, y_val_gwo, param_names
                    )
                    
                    gwo = GreyWolfOptimizer(
                        objective_function=objective_func,
                        n_wolves=25,
                        n_iterations=50,
                        bounds=bounds,
                        verbose=True
                    )
                    
                    best_params, best_fitness = gwo.optimize()
                    
                    gwo_results[name] = {
                        'best_params': best_params,
                        'best_fitness': best_fitness,
                        'gwo_optimizer': gwo
                    }
                    
                    # Create model with optimized complex parameters
                    if class_name == 'QulacsVQC':
                        model = model_class(
                            reps=best_params[0],
                            kernel_scale=best_params[1],
                            regularization=best_params[2]
                        )
                    elif class_name == 'QulacsQNN':
                        model = model_class(
                            reps=best_params[0],
                            learning_rate=best_params[1],
                            rotation_scale=best_params[2]
                        )
                    elif class_name == 'QulacsQDT':
                        model = model_class(
                            layers=best_params[0],
                            rotation_strength=best_params[1],
                            entanglement_strength=best_params[2]
                        )
                    elif class_name == 'QulacsQSVM':
                        model = model_class(
                            max_depth=best_params[0],
                            min_samples_split=best_params[1],
                            quantum_threshold_scale=best_params[2]
                        )
                    
                    # Format params for display (3 decimals)
                    param_display = {k: f"{v:.3f}" for k, v in zip(param_names, best_params)}
                    print(f"      ✓ Optimized: {param_display}")
                else:
                    # Fallback for models without GWO
                    model = model_class()
            else:
                model = model_class()
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
            metrics = calculate_metrics(y_test, y_pred)
            results[name] = metrics
            y_pred_dict[name] = y_pred
            trained_models[name] = model
            
            print(f"✅ F1={metrics['F1-Score']:.4f}, Acc={metrics['Accuracy']:.4f}")

        except Exception as e:
            print(f"❌ Failed: {str(e)[:60]}")
            results[name] = {'Accuracy': 0, 'Precision': 0, 'Recall': 0, 'F1-Score': 0}

    # Save task results immediately
    save_task_results(task_name, results, dataset_name, task_number)

    # GWO plots
    if use_gwo and gwo_results:
        print(f"\n   📊 Generating GWO plots...")
        gwo_save_dir = f'gwo_plots_{dataset_name.lower().replace(" ", "_")}'
        plot_combined_gwo_convergence(gwo_results, task_name, gwo_save_dir)

    return results, y_test, y_pred_dict, trained_models, X_train, X_test, reduced_feature_names


# ==================== MAIN ====================

def main():
    """Main execution"""
    print("\n" + "="*80)
    print("🎯 OPTIMIZED QUANTUM ML CLASSIFICATION")
    print("   Features: Task-by-task saving, QSVM at end, Speed optimized")
    print("="*80)

    # Config - Bank Datasets
    CHURN_PATH = r'D:\Quantum\Dr Raed\1st 2 dataests\1st 2 data_Final\Bank Customer Churn Prediction.csv'
    MARKETING_PATH = r'D:\Quantum\Dr Raed\1st 2 dataests\1st 2 data_Final\bank.csv'
    
    N_COMPONENTS = 5  # Increased for better performance
    
    # Models - QSVM at the end of each task
    quantum_models = {
        'VQC': QulacsVQC,
        'QNN': QulacsQNN,
        'QDT': QulacsQDT,
        'QSVM': QulacsQSVM,  # QSVM at the end
    }

    all_results = []
    
    # Tasks - all quantum models including QSVM at the end
    tasks_config = [
        ("Task 5: Quantum + Classical Data + PCA", quantum_models, False, 'pca', 5),
        ("Task 6: Quantum + Classical Data + LDA", quantum_models, False, 'lda', 6),
        ("Task 7: Quantum + Quantum Data + PCA", quantum_models, True, 'pca', 7),
        ("Task 8: Quantum + Quantum Data + LDA", quantum_models, True, 'lda', 8),
    ]

    # ==================== DATASET 1: CHURN ====================
    print("\n" + "🏦" * 40)
    print("DATASET 1: BANK CUSTOMER CHURN")
    print("🏦" * 40)
    
    best_model_churn = {'f1_score': 0}
    
    try:
        X_churn, y_churn, feature_names_churn = load_bank_customer_churn(CHURN_PATH)
        
        # Run regular tasks (VQC, QNN, QDT)
        for task_name, models, use_quantum, reduction, task_num in tasks_config:
            results, y_test, y_pred_dict, trained_models, X_train, X_test, feat_names = run_classification_task(
                task_name, models, X_churn, y_churn, feature_names_churn,
                use_quantum_data=use_quantum, reduction_method=reduction, 
                n_components=N_COMPONENTS, use_gwo=True, 
                dataset_name='Bank_Churn', task_number=task_num
            )
            
            task_short = f"Task {task_num}: {reduction.upper()}"
            
            for model, metrics in results.items():
                all_results.append({
                    'Dataset': 'Bank Churn',
                    'Task': task_short,
                    'Model': model,
                    **metrics
                })
                
                if metrics['F1-Score'] > best_model_churn['f1_score']:
                    best_model_churn = {
                        'f1_score': metrics['F1-Score'],
                        'dataset': 'Bank Churn',
                        'task': task_name,
                        'model_name': model,
                        'model': trained_models[model],
                        'X_train': X_train,
                        'X_test': X_test,
                        'y_test': y_test,
                        'feature_names': feat_names,
                        'metrics': metrics
                    }
        
        # Best model analysis
        if best_model_churn['f1_score'] > 0:
            print(f"\n{'='*80}")
            print("🏆 BANK CHURN - BEST MODEL")
            print('='*80)
            print(f"   Model: {best_model_churn['model_name']}")
            print(f"   Task: {best_model_churn['task']}")
            print(f"   F1: {best_model_churn['f1_score']:.4f}")
            
            results_dir = 'results_bank_churn'
            os.makedirs(f'{results_dir}/shap_plots', exist_ok=True)
            os.makedirs(f'{results_dir}/confusion_matrices', exist_ok=True)
            
            # Confusion matrix
            y_pred_best = best_model_churn['model'].predict(best_model_churn['X_test'])
            plot_confusion_matrix(
                y_true=best_model_churn['y_test'],
                y_pred=y_pred_best,
                model_name=f"BankChurn_{best_model_churn['model_name']}",
                task_name=f"{best_model_churn['dataset']} - {best_model_churn['task']}",
                save_dir=f'{results_dir}/confusion_matrices'
            )
            
            # SHAP
            if SHAP_AVAILABLE:
                print(f"\n   🔍 Starting SHAP explanation for best model...")
                try:
                    explain_with_shap(
                        model=best_model_churn['model'],
                        X_train=best_model_churn['X_train'],
                        X_test=best_model_churn['X_test'],
                        feature_names=best_model_churn['feature_names'],
                        model_name=f"BankChurn_{best_model_churn['model_name']}",
                        task_name=f"{best_model_churn['dataset']} - {best_model_churn['task']}",
                        save_dir=f'{results_dir}/shap_plots'
                    )
                except Exception as e:
                    print(f"   ❌ SHAP failed: {e}")
                    import traceback
                    traceback.print_exc()
            else:
                print(f"\n   ⚠️ SHAP not available - skipping explanation")
            
            print(f"✅ Churn analysis complete! Results in '{results_dir}/'")
            
    except Exception as e:
        print(f"❌ Churn error: {e}")
        import traceback
        traceback.print_exc()

    # ==================== DATASET 2: MARKETING ====================
    print(f"\n{'='*80}")
    print("🔄 RESETTING FOR MARKETING DATASET")
    print('='*80)
    
    print("\n" + "📈" * 40)
    print("DATASET 2: BANK MARKETING")
    print("📈" * 40)
    
    best_model_marketing = {'f1_score': 0}
    
    try:
        X_marketing, y_marketing, feature_names_marketing = load_bank_marketing(MARKETING_PATH)
        
        # Run regular tasks
        for task_name, models, use_quantum, reduction, task_num in tasks_config:
            results, y_test, y_pred_dict, trained_models, X_train, X_test, feat_names = run_classification_task(
                task_name, models, X_marketing, y_marketing, feature_names_marketing,
                use_quantum_data=use_quantum, reduction_method=reduction, 
                n_components=N_COMPONENTS, use_gwo=True, 
                dataset_name='Bank_Marketing', task_number=task_num
            )
            
            task_short = f"Task {task_num}: {reduction.upper()}"
            
            for model, metrics in results.items():
                all_results.append({
                    'Dataset': 'Bank Marketing',
                    'Task': task_short,
                    'Model': model,
                    **metrics
                })
                
                if metrics['F1-Score'] > best_model_marketing['f1_score']:
                    best_model_marketing = {
                        'f1_score': metrics['F1-Score'],
                        'dataset': 'Bank Marketing',
                        'task': task_name,
                        'model_name': model,
                        'model': trained_models[model],
                        'X_train': X_train,
                        'X_test': X_test,
                        'y_test': y_test,
                        'feature_names': feat_names,
                        'metrics': metrics
                    }
        
        # Best model analysis
        if best_model_marketing['f1_score'] > 0:
            print(f"\n{'='*80}")
            print("🏆 BANK MARKETING - BEST MODEL")
            print('='*80)
            print(f"   Model: {best_model_marketing['model_name']}")
            print(f"   Task: {best_model_marketing['task']}")
            print(f"   F1: {best_model_marketing['f1_score']:.4f}")
            
            results_dir = 'results_bank_marketing'
            os.makedirs(f'{results_dir}/shap_plots', exist_ok=True)
            os.makedirs(f'{results_dir}/confusion_matrices', exist_ok=True)
            
            # Confusion matrix
            y_pred_best = best_model_marketing['model'].predict(best_model_marketing['X_test'])
            plot_confusion_matrix(
                y_true=best_model_marketing['y_test'],
                y_pred=y_pred_best,
                model_name=f"BankMarketing_{best_model_marketing['model_name']}",
                task_name=f"{best_model_marketing['dataset']} - {best_model_marketing['task']}",
                save_dir=f'{results_dir}/confusion_matrices'
            )
            
            # SHAP
            if SHAP_AVAILABLE:
                print(f"\n   🔍 Starting SHAP explanation for best model...")
                try:
                    explain_with_shap(
                        model=best_model_marketing['model'],
                        X_train=best_model_marketing['X_train'],
                        X_test=best_model_marketing['X_test'],
                        feature_names=best_model_marketing['feature_names'],
                        model_name=f"BankMarketing_{best_model_marketing['model_name']}",
                        task_name=f"{best_model_marketing['dataset']} - {best_model_marketing['task']}",
                        save_dir=f'{results_dir}/shap_plots'
                    )
                except Exception as e:
                    print(f"   ❌ SHAP failed: {e}")
                    import traceback
                    traceback.print_exc()
            else:
                print(f"\n   ⚠️ SHAP not available - skipping explanation")
            
            print(f"✅ Marketing analysis complete! Results in '{results_dir}/'")
            
    except Exception as e:
        print(f"❌ Marketing error: {e}")
        import traceback
        traceback.print_exc()

    # ==================== FINAL SUMMARY ====================
    print(f"\n{'='*80}")
    print("📊 COMPREHENSIVE RESULTS")
    print('='*80)

    results_df = pd.DataFrame(all_results)
    
    for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
        if col in results_df.columns:
            results_df[col] = results_df[col].apply(lambda x: f"{x:.4f}")

    print("\n" + results_df.to_string(index=False))

    # Save final results
    output_path = 'results_final_comprehensive.csv'
    results_df.to_csv(output_path, index=False)
    print(f"\n💾 Final results: {output_path}")

    print(f"\n{'='*80}")
    print("✅ COMPLETE!")
    print("="*80)
    print("\n📋 SUMMARY:")
    print("   • Task-by-task saving enabled")
    print("   • QSVM runs at the end of each dataset")
    print("   • Speed optimizations applied")
    print("   • GWO iterations reduced to 10")
    print("   • SHAP samples reduced for speed")
    print("   • QSVM limited to 500 training samples")
    print("\n📁 OUTPUT:")
    print("   task_results/bank_churn/ - Task-specific results")
    print("   task_results/bank_marketing/ - Task-specific results")
    print("   results_bank_churn/ - Best model analysis")
    print("   results_bank_marketing/ - Best model analysis")
    print("   gwo_plots_*/ - GWO convergence plots")
    print("="*80)


if __name__ == "__main__":
    main()